# Day 07 - 1교시: Kafka UI와 Docker Compose
> 웹 UI로 Kafka를 시각적으로 관리하고, Docker Compose로 여러 서비스 한번에 실행하기

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **Docker Compose**로 여러 컨테이너를 한번에 실행할 수 있다
- **Kafka UI**를 통해 클러스터 상태를 시각적으로 확인할 수 있다
- UI에서 토픽을 생성하고 메시지를 조회할 수 있다
- CLI와 UI를 상황에 맞게 선택하여 사용할 수 있다

---

## 📚 Day 06 복습: QuickStart 방식의 한계

```
Day 06에서 사용한 방식:
─────────────────────────────────────────────────────────────
docker run -d --name broker apache/kafka:latest

문제점:
• 컨테이너 하나씩 개별 실행 (여러 개 띄우기 번거로움)
• 명령어를 외워야 함 (kafka-topics.sh, kafka-console-producer.sh ...)
• 전체 구조를 한눈에 파악하기 어려움
• 메시지 내용 확인이 번거로움
```

### 오늘 배울 것

```
1. Docker Compose: 여러 컨테이너를 YAML 파일로 한번에 관리
2. Kafka UI: 웹 브라우저에서 Kafka를 시각적으로 관리
```

---
## 🧭 왜 Kafka UI가 필요한가요?

### CLI만 사용할 때의 어려움



- 터미널(CLI)만 사용할 때:
 - 😢 전체 구조를 한눈에 파악하기 어려움
 - 😢 명령어를 외워야 함
 - 😢 메시지 내용 확인이 번거로움
 - 😢 초보자가 현재 상태를 이해하기 힘듦
- Kafka UI를 함께 사용하면:
 - ✅ 브로커, 토픽, 메시지를 시각적으로 확인
 - ✅ 클릭만으로 토픽 생성/삭제
 - ✅ 메시지 내용을 테이블 형태로 조회
 - ✅ 학습 단계에서 "지금 무슨 일이 일어나는지" 이해하기 쉬움




### Kafka UI란?

> **Kafka UI**: Apache Kafka 클러스터를 웹 브라우저에서 관리하고 모니터링하는 오픈소스 도구

- GitHub: [provectus/kafka-ui](https://github.com/provectus/kafka-ui)
- 무료 오픈소스
- 토픽 관리, 메시지 조회, Consumer 그룹 모니터링 등 지원


---
## 🐳 (복습) Docker Compose란?

### 정의

> **Docker Compose**: 여러 Docker 컨테이너를 정의하고 한번에 실행하는 도구

### docker run vs docker compose 비교

| **구분** | **docker run 방식** (Day 06) | **Docker Compose 방식** (오늘 학습) |
| --- | --- | --- |
| **실행 단위** | 컨테이너 하나씩 개별 실행 | **여러 컨테이너**를 한 번에 실행 |
| **관리 편의성** | 여러 개 실행 시 명령어 반복 입력 | **YAML 파일** 하나로 모든 설정 관리 |
| **네트워크** | 컨테이너 간 연결 설정이 복잡함 | 컨테이너 간 **네트워크 자동 연결** |
| **운영 효율** | 일회성 작업이나 테스트에 적합 | **버전 관리)**(Git 및 협업 가능 |

### 예시 비교

**docker run 방식 (여러 명령어 필요):**
```bash

# 1. 네트워크 생성
docker network create kafka-net

# 2. Kafka 브로커 실행 (리스너 설정 추가)
docker run -d --name broker \
  --network kafka-net \
  -e KAFKA_NODE_ID=1 \
  -e KAFKA_PROCESS_ROLES=broker,controller \
  -e KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093 \
  -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://broker:9092 \
  -e KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER \
  -e KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT \
  -e KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093 \
  -e KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1 \
  -e KAFKA_LOG_DIRS=/tmp/kraft-combined-logs \
  -e CLUSTER_ID=MkU3OEVBNTcwNTJENDM2Qk \
  apache/kafka:latest

# 3. Kafka UI 실행
docker run -d --name kafka-ui \
  --network kafka-net \
  -p 8080:8080 \
  -e KAFKA_CLUSTERS_0_NAME=local \
  -e KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS=broker:9092 \
  provectuslabs/kafka-ui:latest
```

**docker compose 방식 (파일 하나로 관리):**
```bash
# docker-compose.yml 작성 후
docker compose up -d  # 한 줄로 모든 컨테이너 실행!
```

---
## 🚀 실습: Docker Compose로 Kafka + UI 실행하기

### Step 1: 프로젝트 폴더 생성

```bash
# ============================================================
# 프로젝트 폴더 생성 및 이동
# ============================================================

mkdir kafka-ui-demo
cd kafka-ui-demo
```

### Step 2: docker-compose.yml 파일 생성

아래 내용을 `docker-compose.yml` 파일로 저장합니다.

```yaml
# ============================================================
# docker-compose.yml
# Kafka 브로커 + Kafka UI를 함께 실행하는 설정 파일
# ============================================================

services:
  # ─────────────────────────────────────────────────────────
  # Kafka 브로커: 메시지를 저장하고 전달하는 핵심 서버
  # ─────────────────────────────────────────────────────────
  broker:
    image: apache/kafka:latest          # Apache 공식 Kafka 이미지
    container_name: broker              # 컨테이너 이름 고정
    ports:
      - "9092:9092"                     # 호스트:컨테이너 포트 매핑
                                        # → localhost:9092로 접근 가능
    environment:
      # ─── KRaft 모드 설정 (Zookeeper 없이 동작) ───
      KAFKA_NODE_ID: 1                  # 브로커 고유 ID
      KAFKA_PROCESS_ROLES: broker,controller
                                        # 브로커 + 컨트롤러 역할 동시 수행

      # ─── 리스너 설정 (네트워크 연결 방식) ───
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
                                        # 수신 대기할 주소 (모든 인터페이스)
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
                                        # 클라이언트에게 알려줄 주소
                                        # Docker 내부에서는 "broker"로 접근
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
                                        # 컨트롤러 투표 설정 (단일 노드)

      # ─── 토픽 기본 설정 ───
      KAFKA_NUM_PARTITIONS: 3           # 토픽 생성 시 기본 파티션 수
                                        # Day 06에서는 1이었음 → 3으로 변경!
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
                                        # Consumer 그룹 리밸런싱 대기 시간

  # ─────────────────────────────────────────────────────────
  # Kafka UI: 웹 기반 Kafka 관리 도구
  # ─────────────────────────────────────────────────────────
  kafka-ui:
    image: provectuslabs/kafka-ui:latest  # Kafka UI 이미지
    container_name: kafka-ui
    depends_on:
      - broker                          # broker가 먼저 실행된 후 시작
                                        # → 의존성 순서 보장
    ports:
      - "8080:8080"                     # 웹 브라우저에서 접속할 포트
                                        # → http://localhost:8080
    environment:
      KAFKA_CLUSTERS_0_NAME: local-kafka
                                        # UI에 표시될 클러스터 이름
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker:9092
                                        # 연결할 Kafka 브로커 주소
                                        # Docker 내부 네트워크에서 "broker"로 접근
```

### 핵심 설정 설명

| 설정 | 의미 |
|------|------|
| `image` | 사용할 Docker 이미지 |
| `container_name` | 컨테이너 이름 고정 (없으면 랜덤 생성) |
| `ports` | 호스트:컨테이너 포트 매핑 |
| `environment` | 환경 변수로 설정 전달 |
| `depends_on` | 의존성 순서 지정 (먼저 실행될 서비스) |

### Step 3: 실행

```bash
# ============================================================
# Docker Compose로 모든 서비스 실행
# ============================================================

docker compose up -d
```

**명령어 해석:**
- `docker compose up`: docker-compose.yml에 정의된 모든 서비스 실행
- `-d`: Detached 모드 (백그라운드 실행)

### Step 4: 실행 확인

```bash
# 실행 중인 컨테이너 확인
docker ps
```

**예상 출력:**
```
CONTAINER ID   IMAGE                           STATUS         PORTS                    NAMES
a1b2c3d4e5f6   apache/kafka:latest            Up 30 seconds   0.0.0.0:9092->9092/tcp   broker
b2c3d4e5f6g7   provectuslabs/kafka-ui:latest  Up 25 seconds   0.0.0.0:8080->8080/tcp   kafka-ui
```

### 현재 구성도

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Docker Compose 네트워크                            │
│                                                                         │
│   ┌─────────────────────────────────────────────────────────────┐       │
│   │                                                             │       │
│   │   ┌─────────────────────┐      ┌─────────────────────┐     │        │
│   │   │   📦 broker         │      │   📦 kafka-ui       │     │        │
│   │   │                     │      │                     │     │        │
│   │   │   Kafka 브로커        │◀────▶│   웹 UI 서버         │     │        │
│   │   │   포트: 9092         │      │   포트: 8080         │     │        │
│   │   │                     │      │                     │     │        │
│   │   └─────────────────────┘      └─────────────────────┘     │        │
│   │           ▲                              ▲                  │       │
│   └───────────┼──────────────────────────────┼──────────────────┘       │
│               │                              │                          │
│               │ localhost:9092               │ localhost:8080           │
│               │                              │                          │
│   ┌───────────┴──────────────────────────────┴──────────────────┐       │
│   │                      🖥️ 호스트 (내 컴퓨터)                      │       │
│   │                                                             │       │
│   │   터미널: kafka 명령어          브라우저: http://localhost:8080   │       │
│   └─────────────────────────────────────────────────────────────┘       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🌐 Kafka UI 둘러보기

### 브라우저에서 접속

```
http://localhost:8080
```

### UI 화면 구성

![Kafka UI 캡쳐](https://cdn.discordapp.com/attachments/1457516082071081045/1460214915783721143/image.png?ex=69661a9d&is=6964c91d&hm=fcaaf3ce82afbe3ae23070c0db149d175959cc4fdd513f26500c22013343c78a&)

### 주요 메뉴 설명

| 메뉴 | 설명 | CLI 대응 명령어 |
|------|------|----------------|
| **Dashboard** | 클러스터 전체 현황 | `kafka-metadata.sh` |
| **Topics** | 토픽 목록 및 관리 | `kafka-topics.sh --list` |
| **Consumers** | Consumer 그룹 현황 | `kafka-consumer-groups.sh` |
| **Brokers** | 브로커 상세 정보 | - |
| **Metrics** | 성능 지표 (선택사항) | - |

---
## 🎨 실습: UI로 토픽 생성하기

### CLI 방식 vs UI 방식 비교

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  CLI 방식:                                                               │
│  ────────                                                               │
│  docker exec broker /opt/kafka/bin/kafka-topics.sh \                    │
│    --bootstrap-server localhost:9092 \                                  │
│    --create \                                                           │
│    --topic my-topic \                                                   │
│    --partitions 3                                                       │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  UI 방식:                                                                │
│  ────────                                                               │
│  1. Topics 메뉴 클릭                                                     │
│  2. "Add a Topic" 버튼 클릭                                              │
│  3. 토픽 이름 입력: my-topic                                              │
│  4. 파티션 수 선택: 3                                                     │
│  5. "Create Topic" 버튼 클릭                                             │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### UI에서 토픽 생성 단계

**1) Topics 메뉴 클릭 → "Add a Topic" 버튼 클릭**

**2) 토픽 정보 입력:**

![Create a new topic - 2](https://cdn.discordapp.com/attachments/1457516082071081045/1460215407125205044/image.png?ex=69661b12&is=6964c992&hm=2d7a71bd8a4a448e21932729831afe5c35d2faa07479f99615f3489109569fb0&)

| 항목 | 설명 |
|------|------|
| **Topic Name** | 토픽 이름 (영문, 숫자, `-`, `_` 사용) |
| **Partitions** | 파티션 수 (병렬 처리를 위한 분할) |
| **Replication Factor** | 복제본 수 (단일 브로커는 1) |
| **Cleanup Policy** | 메시지 정리 정책 (delete: 시간 경과 후 삭제, compact: 키 기준 보관) |

### CLI로 생성 확인

```bash
# ============================================================
# UI에서 만든 토픽이 CLI에서도 보이는지 확인
# ============================================================

docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic ui-created-topic
```

**예상 출력:**
```
Topic: ui-created-topic   PartitionCount: 3   ReplicationFactor: 1
    Topic: ui-created-topic   Partition: 0    Leader: 1   ...
    Topic: ui-created-topic   Partition: 1    Leader: 1   ...
    Topic: ui-created-topic   Partition: 2    Leader: 1   ...
```

---
## 📨 실습: 메시지 보내고 UI에서 확인하기

### Step 1: CLI로 메시지 보내기

```bash
# ============================================================
# Producer로 JSON 형식 메시지 전송
# ============================================================

docker exec -it broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic ui-created-topic
```

**메시지 입력 (JSON 형식):**
```
> {"orderId": 1, "product": "iPhone", "price": 1200000}
> {"orderId": 2, "product": "MacBook", "price": 2500000}
> {"orderId": 3, "product": "AirPods", "price": 280000}
(Ctrl+C로 종료)
```

### Step 2: UI에서 메시지 조회

**Topics > ui-created-topic > Messages 탭 클릭**

![](https://cdn.discordapp.com/attachments/1457516082071081045/1460216417419919547/image.png?ex=69661c03&is=6964ca83&hm=982664a3b0617a1c9e66d744a3f164de6434efddc72fad3764cd798cf7ca69df&)

### 파티션 분산 시각화

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    토픽: ui-created-topic (파티션 3개)                   │
│                                                                         │
│   Producer ──────────────────────┐                                      │
│   (메시지 전송)                   │                                      │
│                                  ▼                                      │
│                    ┌─────────────────────────┐                          │
│                    │     Kafka 브로커         │                          │
│                    │                         │                          │
│   ┌────────────────┼────────────────┼────────────────┐                  │
│   │                │                │                │                  │
│   │  Partition 0   │  Partition 1   │  Partition 2   │                  │
│   │  ┌──────────┐  │  ┌──────────┐  │  ┌──────────┐  │                  │
│   │  │ orderId:1│  │  │ orderId:2│  │  │ orderId:3│  │                  │
│   │  │ iPhone   │  │  │ MacBook  │  │  │ AirPods  │  │                  │
│   │  └──────────┘  │  └──────────┘  │  └──────────┘  │                  │
│   │                │                │                │                  │
│   └────────────────┴────────────────┴────────────────┘                  │
│                                                                         │
│   💡 Kafka는 메시지를 여러 파티션에 자동으로 분산!                         │
│      → 나중에 여러 Consumer가 병렬로 처리 가능                            │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 📤 실습: UI에서 메시지 직접 전송하기

Kafka UI에서는 **Produce Message** 기능으로 메시지를 직접 전송할 수도 있습니다.

### UI에서 메시지 전송

**Topics > ui-created-topic > "Produce Message" 버튼 클릭**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Topics > ui-created-topic > Produce Message                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Key (optional):    [ order-key-4                          ]           │
│                                                                         │
│  Value:                                                                 │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │ {                                                               │   │
│  │   "orderId": 4,                                                 │   │
│  │   "product": "iPad",                                            │   │
│  │   "price": 900000                                               │   │
│  │ }                                                               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  Partition:    [ Partition #0 ]                               │
│                                                                         │
│                              [ Produce Message ]                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### CLI Consumer로 확인

```bash
# ============================================================
# UI에서 보낸 메시지도 CLI Consumer에서 확인 가능
# ============================================================

docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic ui-created-topic \
  --from-beginning
```

UI에서 보낸 메시지도 Consumer에서 확인됩니다!

---
## 🔄 CLI vs UI 사용 가이드

### 언제 무엇을 사용할까?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  CLI (터미널) 사용이 좋은 경우:                                           │
│  ────────────────────────────                                           │
│  • 스크립트로 자동화할 때                                                 │
│  • 서버에 SSH로만 접속 가능할 때                                          │
│  • 대량의 작업을 반복 처리할 때                                           │
│  • 운영 환경에서 작업할 때                                                │
│                                                                         │
│  UI 사용이 좋은 경우:                                                     │
│  ────────────────────                                                   │
│  • 학습/개발 단계에서 상태 파악할 때                                       │
│  • 메시지 내용을 직관적으로 확인할 때                                      │
│  • 토픽/Consumer 그룹 현황을 모니터링할 때                                │
│  • 명령어가 익숙하지 않을 때                                              │
│                                                                         │
│  💡 결론: 둘 다 익혀두면 상황에 맞게 선택할 수 있음!                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 비교표

| 작업 | CLI | UI |
|------|-----|-----|
| 토픽 생성 | `kafka-topics.sh --create` | Topics > Add a Topic |
| 토픽 목록 | `kafka-topics.sh --list` | Topics 메뉴 |
| 메시지 조회 | `kafka-console-consumer.sh` | Topics > Messages |
| 메시지 전송 | `kafka-console-producer.sh` | Topics > Produce Message |
| Consumer 그룹 확인 | `kafka-consumer-groups.sh` | Consumers 메뉴 |

---
## 🧹 정리하기

### Docker Compose 종료

```bash
# ============================================================
# 모든 서비스 종료 (컨테이너 유지)
# ============================================================
docker compose stop

# ============================================================
# 모든 서비스 종료 + 컨테이너 삭제
# ============================================================
docker compose down

# ============================================================
# 모든 서비스 종료 + 컨테이너 + 볼륨 삭제 (데이터 완전 삭제)
# ============================================================
docker compose down -v
```

| 명령어 | 동작 |
|--------|------|
| `docker compose stop` | 컨테이너 정지 (재시작 가능) |
| `docker compose down` | 컨테이너 삭제 |
| `docker compose down -v` | 컨테이너 + 볼륨(데이터) 삭제 |

---
## ❓ FAQ

**Q1. docker-compose.yml 파일 이름을 바꿀 수 있나요?**

네, `-f` 옵션으로 다른 파일을 지정할 수 있습니다:
```bash
docker compose -f my-config.yml up -d
```

---

**Q2. depends_on이 있으면 broker가 완전히 준비된 후 kafka-ui가 시작되나요?**

`depends_on`은 "시작 순서"만 보장하고, broker가 "준비 완료"될 때까지 기다리지는 않습니다.
완전한 준비 대기가 필요하면 `healthcheck`를 사용합니다.

---

**Q3. Kafka UI 외에 다른 UI 도구는 없나요?**

있습니다:
- **Confluent Control Center**: Confluent 상용 제품
- **AKHQ**: 또 다른 오픈소스 Kafka UI
- **Kafdrop**: 경량 Kafka UI

---

**Q4. docker compose up과 docker-compose up의 차이는?**

- `docker compose` (공백): Docker 최신 버전의 통합 명령어 (권장)
- `docker-compose` (하이픈): 예전 독립 실행 파일 방식

기능은 동일하지만, 최신 버전에서는 `docker compose`를 권장합니다.

---

**Q5. KAFKA_ADVERTISED_LISTENERS가 broker:9092인 이유는?**

Docker Compose 내부에서는 서비스 이름(broker)이 DNS로 등록됩니다.
kafka-ui가 broker에 연결할 때 `broker:9092`로 접근합니다.
이 설정이 잘못되면 UI에서 연결 오류가 발생합니다.

---
## 📝 퀴즈

### Q1. Docker Compose의 주요 장점이 아닌 것은?

- A) 여러 컨테이너를 한번에 실행
- B) 설정을 YAML 파일로 관리
- C) 컨테이너 간 네트워크 자동 연결
- D) 컨테이너 이미지 자동 빌드

<details>
<summary>정답 보기</summary>

**정답: D) 컨테이너 이미지 자동 빌드**

이미지 빌드는 가능하지만 "주요 장점"으로 보기 어렵습니다.
Docker Compose의 핵심 장점은 여러 컨테이너 관리와 네트워크 설정입니다.
</details>

---

### Q2. depends_on의 역할은?

- A) 컨테이너 간 네트워크 연결
- B) 서비스 시작 순서 지정
- C) 환경 변수 전달
- D) 포트 매핑 설정

<details>
<summary>정답 보기</summary>

**정답: B) 서비스 시작 순서 지정**

`depends_on: - broker`는 kafka-ui가 broker 이후에 시작하도록 합니다.
단, "완전히 준비된 후"가 아닌 "시작 순서"만 보장합니다.
</details>

---

### Q3. Docker Compose로 모든 서비스를 종료하고 컨테이너를 삭제하는 명령어는?

- A) docker compose stop
- B) docker compose down
- C) docker compose rm
- D) docker compose kill

<details>
<summary>정답 보기</summary>

**정답: B) docker compose down**

`stop`은 컨테이너를 정지만 하고, `down`은 정지 + 삭제합니다.
`down -v`를 사용하면 볼륨(데이터)까지 삭제합니다.
</details>

---

### Q4. Kafka UI에서 토픽에 저장된 메시지를 확인하는 메뉴는?

- A) Dashboard
- B) Brokers
- C) Topics > Messages
- D) Consumers

<details>
<summary>정답 보기</summary>

**정답: C) Topics > Messages**

Topics 메뉴에서 토픽을 선택한 후 Messages 탭에서 저장된 메시지를 확인할 수 있습니다.
</details>

---
## 📋 과제

### 과제 1: 토픽 생성 및 메시지 전송 (난이도: ⭐)

1. Docker Compose로 Kafka + UI를 실행하세요
2. UI에서 `products` 토픽을 파티션 5개로 생성하세요
3. UI의 "Produce Message" 기능으로 3개의 상품 정보를 JSON 형식으로 전송하세요
4. Messages 탭에서 메시지들이 어떤 파티션에 저장되었는지 확인하세요

---

### 과제 2: CLI와 UI 연동 확인 (난이도: ⭐⭐)

1. CLI로 `cli-topic` 토픽을 생성하세요
2. UI에서 해당 토픽이 보이는지 확인하세요
3. CLI Producer로 메시지 5개를 전송하세요
4. UI에서 메시지 내용을 확인하세요
5. UI에서 메시지 1개를 추가로 전송하세요
6. CLI Consumer로 모든 메시지(6개)가 조회되는지 확인하세요

<details>
<summary>💡 힌트</summary>

```bash
# CLI로 토픽 생성
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --create --topic cli-topic

# CLI Producer
docker exec -it broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic cli-topic

# CLI Consumer
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 --topic cli-topic --from-beginning
```
</details>

---

### 과제 3: Consumer 그룹 확인하기 (난이도: ⭐⭐⭐)

1. CLI로 Consumer를 `--group my-group` 옵션과 함께 실행하세요:
   ```bash
   docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
     --bootstrap-server localhost:9092 \
     --topic ui-created-topic \
     --from-beginning \
     --group my-group
   ```
2. UI의 "Consumers" 메뉴에서 `my-group`이 표시되는지 확인하세요
3. Consumer를 종료했다가 다시 실행해보고, 메시지가 다시 출력되는지 확인하세요

> 💡 **힌트**: Consumer 그룹을 사용하면 Kafka가 읽은 위치(offset)를 기억합니다!

---
## 📋 핵심 요약

### Docker Compose

| 항목 | 설명 |
|------|------|
| **역할** | 여러 컨테이너를 YAML 파일로 한번에 관리 |
| **파일명** | `docker-compose.yml` |
| **실행** | `docker compose up -d` |
| **종료** | `docker compose down` |
| **장점** | 설정 파일화, 네트워크 자동 연결, 버전 관리 |

### Kafka UI

| 항목 | 설명 |
|------|------|
| **접속** | http://localhost:8080 |
| **주요 메뉴** | Dashboard, Topics, Consumers, Brokers |
| **토픽 생성** | Topics > Add a Topic |
| **메시지 조회** | Topics > [토픽명] > Messages |
| **메시지 전송** | Topics > [토픽명] > Produce Message |

### CLI vs UI

| 상황 | 추천 |
|------|------|
| 학습/개발 단계 | UI |
| 스크립트 자동화 | CLI |
| 메시지 내용 확인 | UI |
| 운영 환경 작업 | CLI |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Docker Compose로 여러 컨테이너를 실행할 수 있나요? | ☐ |
| docker-compose.yml 파일의 기본 구조를 이해했나요? | ☐ |
| Kafka UI에 접속하여 토픽을 생성할 수 있나요? | ☐ |
| UI에서 메시지를 조회하고 전송할 수 있나요? | ☐ |
| CLI와 UI의 사용 상황을 구분할 수 있나요? | ☐ |

---


# Day 07 - 2교시: Kafka 파티션 실습
> Key 유무에 따른 메시지 분배를 CLI와 Kafka UI로 직접 확인하기

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **Partition**의 역할과 메시지 분배 원리를 설명할 수 있다
- Key 없이 전송할 때 **라운드로빈** 분배를 확인할 수 있다
- Key를 지정하면 **같은 Key = 같은 Partition**임을 확인할 수 있다
- Kafka UI에서 파티션별 메시지 분포를 시각적으로 확인할 수 있다

---

## 📚 Day 06 복습: Partition과 Key

### Partition이란?

> **Partition = Topic을 쪼갠 조각들**

```
Topic: orders (파티션 3개)
┌─────────────────────────────────────────────────────┐
│                                                     │
│   Partition 0    Partition 1    Partition 2        │
│   ┌─────────┐    ┌─────────┐    ┌─────────┐        │
│   │ msg-0   │    │ msg-1   │    │ msg-2   │        │
│   │ msg-3   │    │ msg-4   │    │ msg-5   │        │
│   │ msg-6   │    │ msg-7   │    │ msg-8   │        │
│   └─────────┘    └─────────┘    └─────────┘        │
│                                                     │
└─────────────────────────────────────────────────────┘
```

### 왜 Partition으로 나누나요?

| 이유 | 설명 |
|------|------|
| **병렬 처리** | 여러 Consumer가 동시에 읽을 수 있음 |
| **확장성** | Partition을 늘려 처리량 증가 |
| **순서 보장** | Partition 내에서는 순서 보장 |

### Key가 있을 때 vs 없을 때

| Key 상태 | 분배 방식 | 특징 |
|----------|-----------|------|
| **Key = null** | 라운드로빈 | 파티션에 고르게 분배 |
| **Key 지정** | 해시 기반 | 같은 Key → 항상 같은 Partition |

> 💡 **핵심**: 같은 Key를 가진 메시지는 반드시 같은 Partition에 저장됩니다!

---
## 🛠️ 실습 환경 확인

### Docker Compose 실행 확인

1교시에서 실행한 환경이 동작 중인지 확인합니다.

```bash
# ============================================================
# 컨테이너 상태 확인
# ============================================================
docker ps
```

**예상 출력:**
```
CONTAINER ID   IMAGE                           STATUS         PORTS                    NAMES
xxxxxxxxxxxx   apache/kafka:latest            Up ...         0.0.0.0:9092->9092/tcp   broker
xxxxxxxxxxxx   provectuslabs/kafka-ui:latest  Up ...         0.0.0.0:8080->8080/tcp   kafka-ui
```

실행 중이 아니라면:
```bash
cd kafka-ui-demo
docker compose up -d
```

---
## 🧪 실습 1: 파티션 3개 토픽 생성

### Step 1: 실습용 토픽 생성

```bash
# ============================================================
# 파티션 3개짜리 토픽 생성
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic partition-demo \
  --partitions 3
```

### Step 2: 토픽 정보 확인

```bash
# ============================================================
# 생성된 토픽의 파티션 구조 확인
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic partition-demo
```

**예상 출력:**
```
Topic: partition-demo   TopicId: xxxxx   PartitionCount: 3   ReplicationFactor: 1
    Topic: partition-demo   Partition: 0    Leader: 1   Replicas: 1   Isr: 1
    Topic: partition-demo   Partition: 1    Leader: 1   Replicas: 1   Isr: 1
    Topic: partition-demo   Partition: 2    Leader: 1   Replicas: 1   Isr: 1
```

| 항목 | 의미 |
|------|------|
| **PartitionCount: 3** | 파티션 3개 |
| **Partition: 0, 1, 2** | 파티션 번호 (0부터 시작) |
| **Leader: 1** | 리더 브로커 ID (현재 브로커 1개라 모두 1) |
| **Replicas: 1** | 복제본 위치 (브로커 ID 목록) |
| **Isr: 1** | 동기화된 복제본 (In-Sync Replicas) |

---
## 🧪 실습 2: Key 없이 메시지 전송 (라운드로빈)

### CLI 파이프로 메시지 전송하기

여러 메시지를 한 번에 전송할 때 **Heredoc**을 사용하면 편리합니다.

```bash
# ============================================================
# Key 없이 메시지 6개 전송
# (Sticky Partitioner로 인해 같은 파티션에 집중될 수 있음)
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo << 'EOF'
주문1-아이폰
주문2-맥북
주문3-아이패드
주문4-에어팟
주문5-애플워치
주문6-맥미니
EOF
```

**명령어 해석:**

| 부분 | 설명 |
|------|------|
| `<< 'EOF' ... EOF` | Heredoc - 여러 줄 입력을 전달 |
| `docker exec -i` | `-i` = 표준입력을 컨테이너로 전달 |
| `kafka-console-producer.sh` | Kafka 메시지 전송 CLI 도구 |

### 라운드로빈 분배란?

```
Key가 없을 때 메시지 분배 방식:

메시지 전송 순서:
┌─────────┐
│ 주문1   │ ─────▶ Partition 0
│ 주문2   │ ─────▶ Partition 1
│ 주문3   │ ─────▶ Partition 2
│ 주문4   │ ─────▶ Partition 0  ← 다시 0번부터
│ 주문5   │ ─────▶ Partition 1
│ 주문6   │ ─────▶ Partition 2
└─────────┘

결과: 각 파티션에 2개씩 고르게 분배!
```

> ⚠️ **참고**: 실제로는 정확한 라운드로빈이 아닐 수 있습니다.
> Kafka Producer는 성능을 위해 **배치(batch)** 단위로 전송하며,
> 배치 내 메시지들이 같은 파티션으로 갈 수 있습니다.

### Kafka UI에서 확인하기

**http://localhost:8080 접속 → Topics → partition-demo → Messages**

1. **전체 메시지 확인**: Messages 탭에서 6개 메시지 확인
2. **파티션별 필터링**: Partition 드롭다운에서 각 파티션 선택

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Kafka UI - Topics > partition-demo > Messages                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Partition: [ All ▼ ]  ← 여기서 0, 1, 2 각각 선택해보기                   │
│                                                                         │
│  ┌──────────┬───────────┬──────────────┬─────────────────────┐         │
│  │ Offset   │ Partition │ Key          │ Value               │         │
│  ├──────────┼───────────┼──────────────┼─────────────────────┤         │
│  │ 0        │ 0         │ -            │ 주문1-아이폰          │         │
│  │ 1        │ 0         │ -            │ 주문4-에어팟          │         │
│  │ 0        │ 1         │ -            │ 주문2-맥북           │         │
│  │ 1        │ 1         │ -            │ 주문5-애플워치         │         │
│  │ 0        │ 2         │ -            │ 주문3-아이패드         │         │
│  │ 1        │ 2         │ -            │ 주문6-맥미니          │         │
│  └──────────┴───────────┴──────────────┴─────────────────────┘         │
│                                                                         │
│  💡 Key가 "-"로 표시 = Key 없음 (null)                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### CLI로 파티션별 메시지 확인

```bash
# ============================================================
# 특정 파티션의 메시지만 조회 (파티션 0번)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --partition 0 \
  --from-beginning
```

**각 파티션별로 실행해보기:**
- `--partition 0`: 파티션 0의 메시지
- `--partition 1`: 파티션 1의 메시지
- `--partition 2`: 파티션 2의 메시지

---
## 🧪 실습 3: Key를 지정하여 메시지 전송

### Key:Value 형식으로 전송하기

Key를 지정하려면 `--property` 옵션을 사용합니다.

```bash
# ============================================================
# Key 있는 메시지 전송 (key:value 형식)
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --property parse.key=true \
  --property key.separator=: << 'EOF'
user-A:주문-A1
user-B:주문-B1
user-A:주문-A2
user-C:주문-C1
user-B:주문-B2
user-A:주문-A3
EOF
```

**옵션 설명:**

| 옵션 | 설명 |
|------|------|
| `--property parse.key=true` | 입력에서 Key를 파싱하도록 설정 |
| `--property key.separator=:` | Key와 Value를 `:` 문자로 구분 |

**전송한 메시지:**

| 순서 | Key | Value |
|------|-----|-------|
| 1 | user-A | 주문-A1 |
| 2 | user-B | 주문-B1 |
| 3 | user-A | 주문-A2 |
| 4 | user-C | 주문-C1 |
| 5 | user-B | 주문-B2 |
| 6 | user-A | 주문-A3 |

### 예상 결과: 같은 Key → 같은 Partition

```
Key 기반 파티션 분배:

┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│   Key: user-A  ──────┐                                                  │
│                      │    ┌─────────────┐                               │
│   Key: user-A  ──────┼──▶ │ Partition X │  user-A의 모든 메시지          │
│                      │    │ 주문-A1      │                               │
│   Key: user-A  ──────┘    │ 주문-A2      │                               │
│                           │ 주문-A3      │                               │
│                           └─────────────┘                               │
│                                                                         │
│   Key: user-B  ──────┐    ┌─────────────┐                               │
│                      ├──▶ │ Partition Y │  user-B의 모든 메시지          │
│   Key: user-B  ──────┘    │ 주문-B1      │                               │
│                           │ 주문-B2      │                               │
│                           └─────────────┘                               │
│                                                                         │
│   Key: user-C  ────────▶  ┌─────────────┐                               │
│                           │ Partition Z │  user-C의 메시지               │
│                           │ 주문-C1      │                               │
│                           └─────────────┘                               │
│                                                                         │
│   💡 같은 Key를 가진 메시지는 반드시 같은 Partition에 저장!               │
│   💡 어떤 파티션에 갈지는 Key의 해시값으로 결정 (예측 불가)                 │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Kafka UI에서 Key 확인하기

**http://localhost:8080 → Topics → partition-demo → Messages**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Messages (새로 추가된 6개)                                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌──────────┬───────────┬──────────────┬─────────────────────┐         │
│  │ Offset   │ Partition │ Key          │ Value               │         │
│  ├──────────┼───────────┼──────────────┼─────────────────────┤         │
│  │ 2        │ 0         │ user-A       │ 주문-A1              │         │
│  │ 3        │ 0         │ user-A       │ 주문-A2              │         │
│  │ 4        │ 0         │ user-A       │ 주문-A3              │  ← 모두  │
│  │ 2        │ 1         │ user-B       │ 주문-B1              │  같은    │
│  │ 3        │ 1         │ user-B       │ 주문-B2              │  파티션  │
│  │ 2        │ 2         │ user-C       │ 주문-C1              │         │
│  └──────────┴───────────┴──────────────┴─────────────────────┘         │
│                                                                         │
│  ✅ user-A의 3개 메시지 → 모두 같은 파티션 (예: 0)                        │
│  ✅ user-B의 2개 메시지 → 모두 같은 파티션 (예: 1)                        │
│  ✅ user-C의 1개 메시지 → 다른 파티션 (예: 2)                             │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

> ⚠️ **주의**: 실제 파티션 번호는 Key의 해시값에 따라 달라집니다.
> 중요한 것은 **같은 Key가 같은 파티션에 있는지** 확인하는 것입니다.

---
## 🧪 실습 4: Key 해싱 원리 이해하기

### Key가 파티션에 매핑되는 방식

Kafka는 **murmur2** 해시 알고리즘을 사용합니다.

```python
# Kafka 내부 파티션 결정 공식 (개념적 설명)
target_partition = abs(murmur2(key_bytes)) % num_partitions
```

**예시 (파티션 3개일 때):**

| Key | 해시값 (예시) | 해시값 % 3 | 파티션 |
|-----|--------------|-----------|--------|
| user-A | 12345 | 12345 % 3 = 0 | 0 |
| user-B | 67891 | 67891 % 3 = 1 | 1 |
| user-C | 24680 | 24680 % 3 = 2 | 2 |

### Key 사용이 중요한 상황

| 상황 | Key 예시 | 이유 |
|------|----------|------|
| **사용자별 이벤트** | user_id | 한 사용자의 모든 행동이 순서대로 처리 |
| **주문 처리** | order_id | 한 주문의 생성→결제→배송이 순서 보장 |
| **IoT 센서 데이터** | device_id | 한 기기의 데이터가 순서대로 저장 |
| **채팅 메시지** | room_id | 한 채팅방의 메시지 순서 보장 |

### 순서 보장이 필요한 이유

```
❌ 순서가 보장되지 않으면:

1. 주문 생성 (order-001)
2. 결제 완료 (order-001)
3. 배송 시작 (order-001)

Consumer가 처리할 때:
→ 배송 시작을 먼저 받음 (파티션 2에서)
→ 주문 생성을 나중에 받음 (파티션 0에서)
→ "존재하지 않는 주문인데 배송?" 오류 발생!

✅ 같은 Key(order-001)로 전송하면:
→ 모든 이벤트가 같은 파티션에 순서대로 저장
→ Consumer가 순서대로 처리 가능
```

---
## 🧪 실습 5: 대량 메시지로 분배 확인

### 메시지 파일 생성 후 전송

더 많은 메시지로 분배 패턴을 확인해봅시다.

```bash
# ============================================================
# 메시지 파일 생성 (20개 메시지)
# ============================================================
cat << 'EOF' > /tmp/messages.txt
customer-A:order-A01
customer-B:order-B01
customer-C:order-C01
customer-A:order-A02
customer-B:order-B02
customer-A:order-A03
customer-C:order-C02
customer-D:order-D01
customer-A:order-A04
customer-B:order-B03
customer-E:order-E01
customer-A:order-A05
customer-D:order-D02
customer-C:order-C03
customer-B:order-B04
customer-A:order-A06
customer-E:order-E02
customer-D:order-D03
customer-C:order-C04
customer-B:order-B05
EOF

# 파일 내용 확인
cat /tmp/messages.txt
```

### 파일에서 메시지 전송

```bash
# ============================================================
# 파일 내용을 Kafka로 전송
# ============================================================
cat /tmp/messages.txt | docker exec -i broker \
  /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --property parse.key=true \
  --property key.separator=:
```

> 💡 파일에서 읽어오는 `cat file | docker exec -i` 방식은 정상 동작합니다.

### 파티션별 메시지 개수 확인

```bash
# ============================================================
# 각 파티션의 메시지 offset 확인 (시작~끝)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-run-class.sh \
  kafka.tools.GetOffsetShell \
  --broker-list localhost:9092 \
  --topic partition-demo
```

**예상 출력:**
```
partition-demo:0:15
partition-demo:1:12
partition-demo:2:5
```

→ 파티션 0에 15개, 파티션 1에 12개, 파티션 2에 5개

> 💡 Key가 있으면 고르게 분배되지 않을 수 있습니다.
> 특정 Key가 많으면 그 Key가 매핑된 파티션에 메시지가 몰립니다.

---
## 🧹 실습 정리

### 실습용 토픽 삭제 (선택)

```bash
# ============================================================
# 토픽 삭제 (다음 실습을 위해 정리)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic partition-demo
```

### 임시 파일 삭제

```bash
rm -f /tmp/messages.txt
```

---
## ❓ FAQ

**Q1. Key 없이 전송하면 항상 라운드로빈으로 분배되나요?**

Kafka 2.4+ 버전에서는 **Sticky Partitioner**가 기본입니다.
배치(batch) 내 메시지들은 같은 파티션으로 전송되어 성능을 높입니다.
따라서 정확한 라운드로빈이 아닐 수 있습니다.

**💡 배치(Batch)란?**

Producer는 메시지를 하나씩 보내지 않고, **여러 메시지를 모아서 한 번에 전송**합니다.
이를 **배치(Batch)**라고 합니다.

```
┌─────────────────────────────────────────────────────────────┐
│  Producer 내부 동작                                          │
│                                                             │
│  메시지1 ─┐                                                 │
│  메시지2 ─┼─▶ [ 배치 버퍼 ] ──(조건 충족)──▶ Kafka Broker   │
│  메시지3 ─┘                                                 │
│                                                             │
│  조건: batch.size 도달 OR linger.ms 경과                     │
└─────────────────────────────────────────────────────────────┘
```

**배치 관련 설정:**
- `batch.size` (기본 16KB): 배치 최대 크기. 이 크기에 도달하면 즉시 전송
- `linger.ms` (기본 0ms): 배치를 얼마나 기다릴지. 0이면 즉시 전송, 5ms면 5ms 동안 모음

**Sticky Partitioner + 배치 동작:**
1. Key가 없는 메시지가 들어오면 **하나의 파티션을 선택**
2. 해당 파티션으로 **배치가 다 찰 때까지** 계속 메시지를 보냄
3. 배치가 전송되면 **다음 파티션으로 변경**

따라서 빠르게 연속으로 메시지를 보내면 같은 파티션에 몰리고,
천천히 보내면 (linger.ms 경과) 파티션이 변경될 수 있습니다.

---

**Q2. 파티션 수를 나중에 늘릴 수 있나요?**

네, 늘릴 수 있습니다. 하지만 **줄일 수는 없습니다**.
```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --alter \
  --topic my-topic \
  --partitions 6
```

> ⚠️ **주의**: 파티션을 늘리면 기존 Key의 파티션 매핑이 변경됩니다!
> 같은 Key가 다른 파티션으로 갈 수 있으므로 순서 보장이 깨집니다.

---

**Q3. Key를 숫자로 사용해도 되나요?**

네, Key는 문자열로 변환되어 해시됩니다.
다만 `1`, `01`, `001`은 다른 Key로 취급되므로 주의하세요.

---

**Q4. 파티션 0에 메시지가 몰리는데, 문제인가요?**

Key 분포에 따라 자연스러운 현상입니다.
특정 Key(예: 인기 상품)의 메시지가 많으면 해당 파티션에 몰립니다.
이를 **Hot Partition** 문제라고 합니다.

해결 방법:
- Key 설계 변경 (예: `product-001-timestamp`)
- 파티션 수 증가
- 커스텀 Partitioner 구현

---

**Q5. 같은 Key인데 다른 파티션으로 가는 경우가 있나요?**

정상적으로는 없습니다. 하지만 다음 경우 발생할 수 있습니다:
- 파티션 수를 변경한 경우 (해시값 % 파티션수가 달라짐)
- 커스텀 Partitioner를 사용하는 경우

---
## 📝 퀴즈

### Q1. Kafka에서 Key가 null인 메시지는 어떻게 파티션에 분배되나요?

- A) 항상 파티션 0으로 전송
- B) 라운드로빈 또는 Sticky 방식으로 분배
- C) 랜덤하게 분배
- D) 가장 메시지가 적은 파티션으로 분배

<details>
<summary>정답 보기</summary>

**정답: B) 라운드로빈 또는 Sticky 방식으로 분배**

Key가 null이면 라운드로빈으로 파티션에 고르게 분배됩니다.
Kafka 2.4+에서는 Sticky Partitioner가 기본으로, 배치 단위로 같은 파티션에 전송합니다.
</details>

---

### Q2. 같은 Key를 가진 메시지들은 어떻게 처리되나요?

- A) 랜덤한 파티션으로 분배
- B) 항상 같은 파티션으로 전송
- C) 가장 빈 파티션으로 전송
- D) 첫 번째 파티션으로만 전송

<details>
<summary>정답 보기</summary>

**정답: B) 항상 같은 파티션으로 전송**

Key의 해시값을 파티션 수로 나눈 나머지로 파티션이 결정됩니다.
같은 Key는 같은 해시값 → 같은 파티션으로 전송됩니다.
</details>

---

### Q3. Key를 사용해야 하는 상황으로 가장 적절한 것은?

- A) 메시지를 최대한 빠르게 전송할 때
- B) 특정 엔티티의 이벤트 순서를 보장해야 할 때
- C) 파티션에 균등하게 분배하고 싶을 때
- D) 메시지 크기를 줄이고 싶을 때

<details>
<summary>정답 보기</summary>

**정답: B) 특정 엔티티의 이벤트 순서를 보장해야 할 때**

같은 Key의 메시지는 같은 파티션에 저장되므로 순서가 보장됩니다.
예: 같은 주문 ID, 같은 사용자 ID의 이벤트는 순서대로 처리되어야 할 때
</details>

---

### Q4. 파티션 수를 3개에서 5개로 늘리면 어떤 문제가 발생할 수 있나요?

- A) 기존 메시지가 삭제됨
- B) 기존 Key의 파티션 매핑이 변경될 수 있음
- C) Consumer가 동작을 멈춤
- D) 브로커가 재시작됨

<details>
<summary>정답 보기</summary>

**정답: B) 기존 Key의 파티션 매핑이 변경될 수 있음**

`hash(key) % 3`과 `hash(key) % 5`는 다른 결과를 줍니다.
따라서 같은 Key라도 새 메시지는 다른 파티션으로 갈 수 있어 순서 보장이 깨집니다.
</details>

---
## 📋 핵심 요약

### Partition 분배 방식

| Key 상태 | 분배 방식 | 순서 보장 |
|----------|-----------|----------|
| Key = null | 라운드로빈/Sticky | X (파티션 간) |
| Key 지정 | 해시 기반 | O (같은 Key 내) |

### CLI 명령어 정리

| 작업 | 명령어 |
|------|--------|
| 토픽 생성 | `kafka-topics.sh --create --partitions 3` |
| 토픽 정보 | `kafka-topics.sh --describe` |
| Key 없이 전송 | `echo "msg" \| kafka-console-producer.sh` |
| Key 있게 전송 | `--property parse.key=true --property key.separator=:` |
| 특정 파티션 조회 | `kafka-console-consumer.sh --partition 0` |

### Key 사용 가이드

| 상황 | Key 사용 | 이유 |
|------|----------|------|
| 순서 보장 필요 | O | 같은 Key = 같은 파티션 = 순서 유지 |
| 최대 처리량 | X | 파티션에 고르게 분배 |
| 로그/메트릭 | X | 순서 중요하지 않음 |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| 파티션 3개인 토픽을 생성할 수 있나요? | ☐ |
| Key 없이 메시지를 전송했을 때 분배 방식을 설명할 수 있나요? | ☐ |
| Key:Value 형식으로 메시지를 전송할 수 있나요? | ☐ |
| Kafka UI에서 파티션별 메시지를 확인할 수 있나요? | ☐ |
| 왜 Key를 사용해야 하는지 예시를 들어 설명할 수 있나요? | ☐ |

---


# Day 07 - 3교시: Consumer 병렬 처리 실습
> 여러 Consumer가 파티션을 나눠 읽는 모습을 직접 확인하기

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **Consumer Group**의 동작 원리를 설명할 수 있다
- 여러 Consumer가 **파티션을 분담**하는 모습을 확인할 수 있다
- Consumer 추가/제거 시 **리밸런싱**을 관찰할 수 있다
- Kafka UI에서 Consumer Group 상태를 모니터링할 수 있다

---

## 📚 Day 06 복습: Consumer Group

### Consumer Group이란?

> **Consumer Group = 같은 역할을 하는 Consumer들의 모임**

같은 `group.id`를 가진 Consumer들은 하나의 그룹으로 묶여서 작업을 분담합니다.

### Partition 할당 규칙

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     Consumer Group: order-processor                     │
│                                                                         │
│   Topic: orders (파티션 3개)                                             │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │  Partition 0 ──────▶ Consumer 1                                 │   │
│   │  Partition 1 ──────▶ Consumer 2                                 │   │
│   │  Partition 2 ──────▶ Consumer 3                                 │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│   💡 규칙: 하나의 파티션은 그룹 내 하나의 Consumer만 읽음                   │
│   💡 반대로: 하나의 Consumer는 여러 파티션을 읽을 수 있음                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Consumer 수 vs Partition 수

| 상황 | 결과 |
|------|------|
| Consumer 수 < Partition 수 | 일부 Consumer가 여러 파티션 담당 |
| Consumer 수 = Partition 수 | 1:1 매핑 (이상적) |
| Consumer 수 > Partition 수 | **일부 Consumer는 놀게 됨!** |

---
## 🛠️ 실습 준비

### Step 1: 실습용 토픽 생성

파티션 3개인 토픽을 새로 생성합니다.

```bash
# ============================================================
# 실습용 토픽 생성 (파티션 3개)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic consumer-demo \
  --partitions 3
```

### Step 2: 테스트 메시지 전송

```bash
# ============================================================
# 테스트 메시지 9개 전송 (파티션별 3개씩 예상)
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
msg-1
msg-2
msg-3
msg-4
msg-5
msg-6
msg-7
msg-8
msg-9
EOF
```

---
## 🧪 실습 1: 단일 Consumer로 읽기

### 터미널에서 Consumer 실행

먼저 Consumer 1개로 모든 메시지를 읽어봅니다.

```bash
# ============================================================
# Terminal 1: Consumer 1개로 모든 파티션 읽기
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group \
  --from-beginning
```

**예상 출력:**
```
msg-1
msg-2
msg-3
msg-4
msg-5
msg-6
msg-7
msg-8
msg-9
```

> `Ctrl+C`로 종료하지 말고 계속 실행 상태를 유지하세요!

### 현재 상태

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     Consumer Group: my-consumer-group                   │
│                                                                         │
│   Topic: consumer-demo                                                  │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │  Partition 0 ──┐                                                │   │
│   │  Partition 1 ──┼──▶ Consumer 1 (Terminal 1)                     │   │
│   │  Partition 2 ──┘                                                │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│   💡 Consumer 1개가 모든 파티션(3개)을 담당                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🧪 실습 2: Consumer 2개로 병렬 처리

### 터미널 2개 열기

**새 터미널(Terminal 2)**을 열고 같은 Consumer Group으로 Consumer를 추가합니다.

```bash
# ============================================================
# Terminal 2: 같은 그룹에 Consumer 추가
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group
```

> 💡 `--from-beginning` 없이 실행 → 새 메시지만 읽음

### 리밸런싱 발생!

Consumer를 추가하면 Kafka가 자동으로 **파티션을 재분배**합니다.

```
┌─────────────────────────────────────────────────────────────────────────┐
│  🔄 리밸런싱 발생!                                                       │
│                                                                         │
│  Before (Consumer 1개):                                                 │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  P0 ──┐                                                         │   │
│  │  P1 ──┼──▶ Consumer 1                                           │   │
│  │  P2 ──┘                                                         │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  After (Consumer 2개):                                                  │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  P0 ──┬──▶ Consumer 1 (Terminal 1)                              │   │
│  │  P1 ──┘                                                         │   │
│  │  P2 ──────▶ Consumer 2 (Terminal 2)                             │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  💡 Consumer 1: 2개 파티션, Consumer 2: 1개 파티션 담당                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 새 메시지 전송하여 확인

**새 터미널(Terminal 3)**에서 메시지를 전송합니다.

```bash
# ============================================================
# Terminal 3: 새 메시지 6개 전송
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
new-1
new-2
new-3
new-4
new-5
new-6
EOF
```

**확인 포인트:**
- Terminal 1과 Terminal 2에 메시지가 **나눠서** 출력됨
- 각 Consumer가 담당하는 파티션의 메시지만 받음

---
## 🧪 실습 3: Consumer 3개로 1:1 매핑

### 세 번째 Consumer 추가

**새 터미널(Terminal 4)**에서 Consumer를 하나 더 추가합니다.

```bash
# ============================================================
# Terminal 4: 세 번째 Consumer 추가
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group
```

### 이상적인 1:1 매핑!

```
┌─────────────────────────────────────────────────────────────────────────┐
│  ✅ 이상적인 상태: Consumer 수 = Partition 수                            │
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Partition 0 ──────▶ Consumer 1 (Terminal 1)                    │   │
│  │  Partition 1 ──────▶ Consumer 2 (Terminal 2)                    │   │
│  │  Partition 2 ──────▶ Consumer 3 (Terminal 4)                    │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  💡 각 Consumer가 정확히 1개의 파티션만 담당                              │
│  💡 부하가 균등하게 분배됨                                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 다시 메시지 전송

```bash
# ============================================================
# Terminal 3: 메시지 추가 전송
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
final-1
final-2
final-3
final-4
final-5
final-6
EOF
```

**확인 포인트:**
- 3개 터미널에 메시지가 **골고루** 분배됨
- 각 Consumer가 약 2개씩 메시지 수신

---
## 🧪 실습 4: Consumer 초과 시 (파티션보다 많을 때)

### 네 번째 Consumer 추가

파티션(3개)보다 Consumer(4개)가 많으면 어떻게 될까요?

```bash
# ============================================================
# Terminal 5: 네 번째 Consumer 추가
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group
```

### 일부 Consumer는 대기 상태!

```
┌─────────────────────────────────────────────────────────────────────────┐
│  ⚠️ Consumer 수 > Partition 수                                          │
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Partition 0 ──────▶ Consumer 1                                 │   │
│  │  Partition 1 ──────▶ Consumer 2                                 │   │
│  │  Partition 2 ──────▶ Consumer 3                                 │   │
│  │                                                                 │   │
│  │         ❌ Consumer 4 ──── (할당된 파티션 없음, 대기 중)           │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  💡 Consumer 4는 메시지를 받지 못함!                                     │
│  💡 다른 Consumer가 장애나면 그때 파티션을 할당받음                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 메시지 전송하여 확인

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
test-1
test-2
test-3
EOF
```

**확인 포인트:**
- Terminal 5 (Consumer 4)에는 **아무 메시지도 출력되지 않음**
- 3개의 Consumer만 메시지를 받음

---
## 🧪 실습 5: Consumer 장애 시 리밸런싱

### Consumer 하나 종료

Terminal 1에서 `Ctrl+C`를 눌러 Consumer 1을 종료합니다.

```
Consumer 1 종료 (Ctrl+C)
```

### 자동 리밸런싱!

```
┌─────────────────────────────────────────────────────────────────────────┐
│  🔄 리밸런싱 발생! (Consumer 1 장애)                                     │
│                                                                         │
│  Before:                                                                │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  P0 ──▶ Consumer 1 ❌ (종료됨)                                   │   │
│  │  P1 ──▶ Consumer 2                                              │   │
│  │  P2 ──▶ Consumer 3                                              │   │
│  │         Consumer 4 (대기 중)                                     │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  After:                                                                 │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  P0 ──▶ Consumer 4 ✅ (새로 할당!)                               │   │
│  │  P1 ──▶ Consumer 2                                              │   │
│  │  P2 ──▶ Consumer 3                                              │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  💡 대기 중이던 Consumer 4가 파티션 0을 할당받음!                         │
│  💡 서비스 중단 없이 장애 복구                                           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 메시지 전송하여 확인

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
after-failure-1
after-failure-2
after-failure-3
EOF
```

**확인 포인트:**
- Terminal 5 (Consumer 4)에 **이제 메시지가 출력됨!**
- 이전에 대기 중이던 Consumer가 파티션을 인계받음

---
## 🌐 Kafka UI에서 Consumer Group 확인

### Consumers 메뉴 접속

**http://localhost:8080 → Consumers**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Kafka UI - Consumers                                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌──────────────────────┬──────────────┬──────────────┬───────────┐    │
│  │ Consumer Group       │ Members      │ Topics       │ Lag       │    │
│  ├──────────────────────┼──────────────┼──────────────┼───────────┤    │
│  │ my-consumer-group    │ 3            │ 1            │ 0         │    │
│  └──────────────────────┴──────────────┴──────────────┴───────────┘    │
│                                                                         │
│  💡 Members: 현재 활성 Consumer 수                                       │
│  💡 Lag: 처리되지 않은 메시지 수 (0이면 모두 처리됨)                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Consumer Group 상세 보기

**my-consumer-group 클릭 → 상세 정보**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Consumer Group: my-consumer-group                                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  State: Stable  (안정 상태)                                              │
│                                                                         │
│  ┌───────────┬────────────────┬────────────┬────────────┬───────────┐  │
│  │ Topic     │ Partition      │ Consumer   │ Offset     │ Lag       │  │
│  ├───────────┼────────────────┼────────────┼────────────┼───────────┤  │
│  │ consumer  │ 0              │ consumer-1 │ 15         │ 0         │  │
│  │ consumer  │ 1              │ consumer-2 │ 12         │ 0         │  │
│  │ consumer  │ 2              │ consumer-3 │ 10         │ 0         │  │
│  └───────────┴────────────────┴────────────┴────────────┴───────────┘  │
│                                                                         │
│  💡 각 파티션이 어떤 Consumer에 할당되었는지 확인 가능                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🔍 CLI로 Consumer Group 상태 확인

### Consumer Group 목록 조회

```bash
# ============================================================
# Consumer Group 목록 확인
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --list
```

**예상 출력:**
```
my-consumer-group
```

### Consumer Group 상세 정보

```bash
# ============================================================
# Consumer Group 상세 정보 (파티션 할당 상태)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group my-consumer-group
```

**예상 출력:**
```
GROUP              TOPIC         PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG   CONSUMER-ID                                HOST            CLIENT-ID
my-consumer-group  consumer-demo  0          15              15              0     console-consumer-xxx  /172.19.0.3     console-consumer
my-consumer-group  consumer-demo  1          12              12              0     console-consumer-yyy  /172.19.0.3     console-consumer
my-consumer-group  consumer-demo  2          10              10              0     console-consumer-zzz  /172.19.0.3     console-consumer
```

**출력 항목 설명:**

| 항목 | 설명 |
|------|------|
| **PARTITION** | 파티션 번호 |
| **CURRENT-OFFSET** | 현재까지 읽은 위치 |
| **LOG-END-OFFSET** | 파티션의 마지막 메시지 위치 |
| **LAG** | 처리되지 않은 메시지 수 (END - CURRENT) |
| **CONSUMER-ID** | Consumer 고유 식별자 |

---
## 🧹 실습 정리

### 모든 Consumer 종료

실행 중인 모든 터미널에서 `Ctrl+C`를 눌러 Consumer를 종료합니다.

### 토픽 삭제 (선택)

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic consumer-demo
```

---
## ❓ FAQ

**Q1. 리밸런싱 중에 메시지 처리가 멈추나요?**

네, 리밸런싱 동안 해당 Consumer Group의 모든 Consumer는 잠시 멈춥니다.
이를 "stop-the-world" 현상이라고 합니다.
Kafka 2.3+에서는 **Incremental Cooperative Rebalancing**으로 영향을 최소화합니다.

---

**Q2. Consumer가 갑자기 죽으면 메시지가 유실되나요?**

기본 설정에서는 유실되지 않습니다.
- Consumer가 죽으면 offset이 커밋되지 않음
- 다른 Consumer가 파티션을 인계받으면 마지막 커밋 위치부터 다시 읽음
- 단, **중복 처리**가 발생할 수 있음 (At Least Once)

---

**Q3. 파티션 수보다 Consumer가 많은 게 문제인가요?**

낭비일 뿐 문제는 아닙니다.
오히려 **고가용성**을 위해 일부러 여분의 Consumer를 띄워두기도 합니다.
활성 Consumer가 죽으면 대기 중인 Consumer가 즉시 인계받습니다.

---

**Q4. 리밸런싱은 언제 발생하나요?**

- Consumer가 그룹에 **참가**할 때
- Consumer가 그룹에서 **이탈**할 때 (정상 종료, 장애)
- Topic의 **파티션 수가 변경**될 때
- Consumer가 **heartbeat를 보내지 못할 때** (session timeout)

---

**Q5. 같은 메시지를 여러 서비스에서 받으려면?**

**다른 Consumer Group**을 사용하세요!
같은 Topic을 여러 Group이 구독하면, 각 Group은 **모든 메시지**를 받습니다.

```
Topic: orders
├── Consumer Group: order-processor → 모든 메시지 수신
├── Consumer Group: analytics       → 모든 메시지 수신
└── Consumer Group: notification    → 모든 메시지 수신
```

---
## 📝 퀴즈

### Q1. Consumer Group 내에서 하나의 파티션은 몇 개의 Consumer가 읽을 수 있나요?

- A) 무제한
- B) 파티션 수만큼
- C) 1개
- D) 2개

<details>
<summary>정답 보기</summary>

**정답: C) 1개**

Consumer Group 내에서 각 파티션은 **하나의 Consumer만** 읽을 수 있습니다.
이를 통해 메시지 중복 처리를 방지하고 순서를 보장합니다.
</details>

---

### Q2. 파티션 3개인 토픽에 Consumer 5개를 연결하면?

- A) 오류가 발생한다
- B) 2개의 Consumer는 대기 상태가 된다
- C) 5개 모두 메시지를 받는다
- D) 가장 먼저 연결한 3개만 메시지를 받는다

<details>
<summary>정답 보기</summary>

**정답: B) 2개의 Consumer는 대기 상태가 된다**

파티션 수보다 Consumer가 많으면 초과된 Consumer는 파티션을 할당받지 못합니다.
다른 Consumer가 장애 발생 시 대기 중인 Consumer가 파티션을 인계받습니다.
</details>

---

### Q3. Consumer가 그룹에 새로 참가하면 어떤 일이 발생하나요?

- A) 아무 일도 없음
- B) 리밸런싱이 발생하여 파티션이 재분배됨
- C) 기존 Consumer들이 종료됨
- D) 새 토픽이 생성됨

<details>
<summary>정답 보기</summary>

**정답: B) 리밸런싱이 발생하여 파티션이 재분배됨**

Consumer가 참가하거나 이탈할 때 Kafka는 파티션을 재분배합니다.
이 과정에서 잠시 메시지 처리가 멈출 수 있습니다.
</details>

---

### Q4. 같은 토픽의 메시지를 두 서비스에서 각각 전체를 받으려면?

- A) Consumer를 2배로 늘린다
- B) 파티션을 2배로 늘린다
- C) 서로 다른 Consumer Group을 사용한다
- D) 토픽을 2개 만든다

<details>
<summary>정답 보기</summary>

**정답: C) 서로 다른 Consumer Group을 사용한다**

각 Consumer Group은 토픽의 모든 메시지를 독립적으로 수신합니다.
Group A와 Group B가 같은 토픽을 구독하면 둘 다 모든 메시지를 받습니다.
</details>

---
## 📋 핵심 요약

### Consumer Group 규칙

| 규칙 | 설명 |
|------|------|
| 1 Partition : 1 Consumer | 파티션은 그룹 내 하나의 Consumer만 읽음 |
| 1 Consumer : N Partitions | 하나의 Consumer는 여러 파티션 가능 |
| Consumer > Partition | 초과 Consumer는 대기 |

### 리밸런싱

| 발생 시점 | 결과 |
|-----------|------|
| Consumer 참가 | 파티션 재분배 |
| Consumer 이탈 | 남은 Consumer에 재분배 |
| 파티션 수 변경 | 전체 재분배 |

### CLI 명령어 정리

| 작업 | 명령어 |
|------|--------|
| Consumer 그룹으로 읽기 | `--group my-group` |
| 그룹 목록 | `kafka-consumer-groups.sh --list` |
| 그룹 상세 | `kafka-consumer-groups.sh --describe --group xxx` |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| 여러 터미널에서 같은 Consumer Group으로 Consumer를 실행할 수 있나요? | ☐ |
| Consumer 수가 파티션 수보다 많을 때 어떻게 되는지 알고 있나요? | ☐ |
| 리밸런싱이 언제 발생하는지 설명할 수 있나요? | ☐ |
| Kafka UI에서 Consumer Group 상태를 확인할 수 있나요? | ☐ |
| CLI로 Consumer Group의 파티션 할당 상태를 조회할 수 있나요? | ☐ |

---


# Day 07 - 4교시: Consumer Group과 Offset 실습
> 메시지 읽기 위치(Offset)를 이해하고 관리하기

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **Offset**의 개념과 역할을 설명할 수 있다
- 같은 Group vs 다른 Group의 **메시지 소비 차이**를 이해한다
- CLI와 UI에서 **Offset을 확인**할 수 있다
- **Offset 리셋**을 통해 메시지를 다시 읽을 수 있다

---

## 📚 Day 06 복습: Offset이란?

### Offset 개념

> **Offset = 파티션 내 메시지의 고유 위치 번호**

```
Partition 0:
┌────┬────┬────┬────┬────┬────┬────┬────┐
│ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │  ← offset
└────┴────┴────┴────┴────┴────┴────┴────┘
  ↑                                    ↑
 가장 오래된 메시지              가장 최신 메시지
```

### Consumer Offset

> **Consumer Offset = Consumer가 마지막으로 읽은 메시지의 위치**

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│   Partition 0:                                                          │
│   ┌────┬────┬────┬────┬────┬────┬────┬────┐                            │
│   │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │                            │
│   └────┴────┴────┴────┴────┴────┴────┴────┘                            │
│                    ↑              ↑                                     │
│              Current Offset   Log End Offset                            │
│              (읽은 위치)       (마지막 메시지)                             │
│                                                                         │
│   💡 LAG = Log End Offset - Current Offset = 7 - 3 = 4                  │
│      (아직 처리하지 않은 메시지 수)                                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Offset이 왜 중요한가요?

| 상황 | Offset 활용 |
|------|-------------|
| Consumer 재시작 | 마지막 읽은 위치부터 이어서 읽기 |
| Consumer 장애 | 다른 Consumer가 같은 위치부터 계속 읽기 |
| 데이터 재처리 | Offset을 되돌려서 과거 데이터 다시 읽기 |

---
## 🛠️ 실습 준비

### Step 1: 실습용 토픽 생성

```bash
# ============================================================
# 실습용 토픽 생성 (파티션 1개 - Offset 흐름 관찰 용이)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic offset-demo \
  --partitions 1
```

### Step 2: 테스트 메시지 전송

```bash
# ============================================================
# 10개 메시지 전송
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo << 'EOF'
msg-00
msg-01
msg-02
msg-03
msg-04
msg-05
msg-06
msg-07
msg-08
msg-09
EOF
```

---
## 🧪 실습 1: 같은 Consumer Group의 Offset 공유

### 첫 번째 Consumer 실행 (Group A)

```bash
# ============================================================
# Terminal 1: Group A의 Consumer
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A \
  --from-beginning
```

**예상 출력:**
```
msg-00
msg-01
msg-02
...
msg-09
```

10개 메시지를 모두 읽었습니다. `Ctrl+C`로 종료합니다.

### 같은 Group A로 다시 Consumer 실행

```bash
# ============================================================
# Terminal 1: 같은 Group A로 다시 Consumer 실행
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A
```

**예상 출력:**
```
(아무것도 출력되지 않음 - 대기 중)
```

### 왜 메시지가 안 나올까요?

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Group A의 Offset 상태                                                  │
│                                                                         │
│   Partition 0:                                                          │
│   ┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐                  │
│   │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │ 8  │ 9  │                  │
│   └────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘                  │
│                                                         ↑               │
│                                                   Committed Offset: 10  │
│                                                                         │
│   💡 첫 번째 Consumer가 offset 10까지 커밋함                              │
│   💡 두 번째 Consumer는 10부터 읽으려 하지만 새 메시지 없음                 │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

> Kafka는 Consumer Group별로 Offset을 기억합니다!
> 같은 Group이면 이전 Consumer가 읽은 다음부터 시작합니다.

---
## 🧪 실습 2: 다른 Consumer Group은 처음부터 읽기

### 새로운 Group B로 Consumer 실행

```bash
# ============================================================
# Terminal 2: Group B의 Consumer (새 그룹)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-B \
  --from-beginning
```

**예상 출력:**
```
msg-00
msg-01
msg-02
...
msg-09
```

### Group별 Offset 독립성

```
┌─────────────────────────────────────────────────────────────────────────┐
│  각 Consumer Group은 독립적인 Offset을 가짐                               │
│                                                                         │
│   Topic: offset-demo                                                    │
│   ┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐                  │
│   │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │ 8  │ 9  │                  │
│   └────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘                  │
│                                                         ↑               │
│                                         Group A: offset 10 (완료)        │
│                                         Group B: offset 10 (완료)        │
│                                         Group C: offset 0 (아직 안 읽음)  │
│                                                                         │
│   💡 같은 Topic을 여러 서비스가 각자 읽을 수 있음!                          │
│   💡 예: 주문 처리 서비스, 분석 서비스, 알림 서비스                         │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🧪 실습 3: Offset 상태 확인하기

### CLI로 Offset 확인

```bash
# ============================================================
# Consumer Group의 Offset 상태 확인
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A
```

**예상 출력:**
```
GROUP    TOPIC       PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG  CONSUMER-ID  HOST  CLIENT-ID
group-A  offset-demo  0          10              10              0    -            -     -
```

**출력 항목 설명:**

| 항목 | 값 | 의미 |
|------|-----|------|
| **CURRENT-OFFSET** | 10 | 마지막으로 읽은 위치 |
| **LOG-END-OFFSET** | 10 | 파티션의 마지막 메시지 위치 |
| **LAG** | 0 | 처리 안 된 메시지 수 (0 = 모두 읽음) |
| **CONSUMER-ID** | - | 현재 연결된 Consumer 없음 |

### 새 메시지 추가 후 LAG 확인

```bash
# ============================================================
# 새 메시지 5개 추가
# ============================================================
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo << 'EOF'
new-01
new-02
new-03
new-04
new-05
EOF

# Offset 상태 다시 확인
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A
```

**예상 출력:**
```
GROUP    TOPIC        PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
group-A  offset-demo  0          10              15              5
```

> **LAG = 5** → 아직 읽지 않은 메시지 5개!

---
## 🧪 실습 4: Kafka UI에서 Offset 확인

### Consumers 메뉴에서 확인

**http://localhost:8080 → Consumers → group-A 클릭**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Consumer Group: group-A                                                │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  State: Empty  (현재 연결된 Consumer 없음)                                │
│                                                                         │
│  ┌───────────┬────────────┬────────────────┬───────────────┬─────────┐ │
│  │ Topic     │ Partition  │ Current Offset │ End Offset    │ Lag     │ │
│  ├───────────┼────────────┼────────────────┼───────────────┼─────────┤ │
│  │ offset-   │ 0          │ 10             │ 15            │ 5       │ │
│  │ demo      │            │                │               │         │ │
│  └───────────┴────────────┴────────────────┴───────────────┴─────────┘ │
│                                                                         │
│  💡 Lag 5: 5개 메시지가 아직 처리되지 않음                                 │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Consumer 연결 시 변화 확인

터미널에서 Group A Consumer를 다시 실행하고 UI를 새로고침하면:
- State가 **Stable**로 변경
- Consumer-ID가 표시됨
- Lag가 **0**으로 감소

---
## 🧪 실습 5: Offset 리셋하기

### 언제 Offset 리셋이 필요할까요?

| 상황 | 리셋 대상 |
|------|-----------|
| 버그 수정 후 재처리 | 특정 시점부터 다시 |
| 데이터 마이그레이션 | 처음부터 다시 |
| 테스트/개발 | 처음부터 다시 |
| 오래된 데이터 건너뛰기 | 최신으로 이동 |

### Step 1: 현재 Offset 확인

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A
```

### Step 2: Offset을 처음(earliest)으로 리셋

> ⚠️ **중요**: Consumer가 연결되어 있으면 리셋이 실패합니다!
> 먼저 모든 Consumer를 종료하세요.

```bash
# ============================================================
# Offset 리셋 (모든 파티션을 처음으로)
# --dry-run: 실제 실행 전 미리보기
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-earliest \
  --dry-run
```

**예상 출력 (dry-run):**
```
GROUP    TOPIC        PARTITION  NEW-OFFSET
group-A  offset-demo  0          0
```

### Step 3: 실제 리셋 실행

```bash
# ============================================================
# Offset 리셋 실제 실행 (--execute 옵션)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-earliest \
  --execute
```

**예상 출력:**
```
GROUP    TOPIC        PARTITION  NEW-OFFSET
group-A  offset-demo  0          0
```

### Step 4: 리셋 확인

```bash
# Consumer 다시 실행 - 처음부터 모든 메시지 출력!
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A
```

**예상 출력:**
```
msg-00
msg-01
...
msg-09
new-01
new-02
...
new-05
```

> 처음부터 15개 메시지 모두 다시 읽기!

---
## 🧪 실습 6: 다양한 Offset 리셋 옵션

### 리셋 옵션 종류

| 옵션 | 설명 | 사용 예시 |
|------|------|----------|
| `--to-earliest` | 가장 처음으로 | 전체 재처리 |
| `--to-latest` | 가장 최신으로 | 과거 데이터 무시 |
| `--to-offset <N>` | 특정 offset으로 | 정확한 위치 지정 |
| `--shift-by <N>` | 현재 위치에서 N만큼 이동 | 일부 메시지 건너뛰기 |
| `--to-datetime` | 특정 시간으로 | 시간 기준 재처리 |

### 예시: 최신(latest)으로 이동

```bash
# ============================================================
# 과거 메시지 무시하고 최신부터 읽기
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-latest \
  --execute
```

### 예시: 특정 offset으로 이동

```bash
# ============================================================
# offset 5부터 읽기 (0~4는 건너뜀)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-offset 5 \
  --execute
```

### 예시: 현재 위치에서 뒤로 3칸

```bash
# ============================================================
# 현재 위치에서 3개 메시지 뒤로 이동 (재처리)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --shift-by -3 \
  --execute
```

---
## 🧪 실습 7: LAG 모니터링

### LAG란?

> **LAG = 아직 처리하지 않은 메시지 수**

LAG가 계속 증가하면 Consumer가 Producer의 속도를 따라가지 못하는 것입니다.

```
┌─────────────────────────────────────────────────────────────────────────┐
│  LAG 모니터링                                                           │
│                                                                         │
│   정상 상태:                                                             │
│   Producer ─────────────▶ Topic ─────────────▶ Consumer                 │
│              100 msg/s           100 msg/s                              │
│              LAG ≈ 0 (안정)                                              │
│                                                                         │
│   문제 상태:                                                             │
│   Producer ─────────────▶ Topic ─────────────▶ Consumer                 │
│              100 msg/s           50 msg/s                               │
│              LAG ↑ (계속 증가!)                                          │
│                                                                         │
│   💡 LAG 증가 시 조치:                                                   │
│      • Consumer 수 증가                                                  │
│      • Consumer 처리 로직 최적화                                         │
│      • 파티션 수 증가 (장기적)                                            │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 실습: LAG 발생시키기

```bash
# ============================================================
# 1. 먼저 Offset을 최신으로 이동
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-latest \
  --execute

# ============================================================
# 2. 새 메시지 대량 전송 (Consumer 연결 없이)
# ============================================================
for i in $(seq 1 20); do
  echo "lag-test-$i"
done | docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo

# ============================================================
# 3. LAG 확인
# ============================================================
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A
```

**예상 출력:**
```
GROUP    TOPIC        PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
group-A  offset-demo  0          15              35              20
```

> **LAG = 20** → 20개 메시지가 대기 중!

---
## 🧹 실습 정리

### 토픽 삭제 (선택)

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic offset-demo
```

### Consumer Group 삭제 (선택)

```bash
# Consumer Group 삭제 (연결된 Consumer가 없어야 함)
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --group group-A

docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --group group-B
```

---
## ❓ FAQ

**Q1. Offset은 어디에 저장되나요?**

Kafka의 내부 토픽 `__consumer_offsets`에 저장됩니다.
이 토픽은 자동으로 생성되며 직접 수정하면 안 됩니다.

---

**Q2. Consumer가 메시지를 읽으면 자동으로 Offset이 커밋되나요?**

기본 설정(`enable.auto.commit=true`)에서는 5초마다 자동 커밋됩니다.
수동 커밋도 가능합니다 (프로그래밍 시 `commit()` 호출).

---

**Q3. Offset 리셋 시 Consumer가 연결되어 있으면?**

리셋이 실패합니다. 반드시 모든 Consumer를 종료한 후 리셋해야 합니다.

---

**Q4. --from-beginning 옵션이 안 먹히는 것 같아요**

`--from-beginning`은 **해당 Group의 Offset이 없을 때만** 작동합니다.
이미 Offset이 있으면 저장된 위치부터 읽습니다.
처음부터 다시 읽으려면 Offset 리셋이 필요합니다.

---

**Q5. LAG가 계속 0인데 메시지가 잘 처리되는 건가요?**

네! LAG 0은 Consumer가 Producer를 잘 따라잡고 있다는 의미입니다.
이상적인 상태입니다.

---
## 📝 퀴즈

### Q1. 같은 Consumer Group으로 Consumer를 다시 실행하면?

- A) 처음부터 모든 메시지를 다시 읽음
- B) 이전에 읽은 다음 메시지부터 읽음
- C) 가장 최신 메시지만 읽음
- D) 오류가 발생함

<details>
<summary>정답 보기</summary>

**정답: B) 이전에 읽은 다음 메시지부터 읽음**

Kafka는 Consumer Group별로 Offset을 저장합니다.
같은 Group이면 마지막으로 커밋한 Offset 다음부터 읽습니다.
</details>

---

### Q2. LAG가 계속 증가하고 있다면 어떤 문제일까요?

- A) Producer가 너무 느림
- B) Consumer가 Producer 속도를 따라가지 못함
- C) 브로커가 다운됨
- D) 네트워크 문제

<details>
<summary>정답 보기</summary>

**정답: B) Consumer가 Producer 속도를 따라가지 못함**

LAG = (Log End Offset) - (Current Offset)
LAG가 증가하면 Consumer의 처리 속도가 Producer보다 느린 것입니다.
Consumer 수를 늘리거나 처리 로직을 최적화해야 합니다.
</details>

---

### Q3. Offset을 처음으로 리셋하려면 어떤 옵션을 사용하나요?

- A) --to-latest
- B) --to-earliest
- C) --to-beginning
- D) --reset-all

<details>
<summary>정답 보기</summary>

**정답: B) --to-earliest**

`--to-earliest`는 파티션의 가장 처음(offset 0)으로 이동합니다.
`--to-latest`는 가장 최신(마지막)으로 이동합니다.
</details>

---

### Q4. Offset 리셋 전에 반드시 해야 할 것은?

- A) 토픽 삭제
- B) 브로커 재시작
- C) 해당 Group의 모든 Consumer 종료
- D) 새 메시지 전송

<details>
<summary>정답 보기</summary>

**정답: C) 해당 Group의 모든 Consumer 종료**

Consumer가 연결된 상태에서는 Offset 리셋이 실패합니다.
모든 Consumer를 종료한 후 리셋해야 합니다.
</details>

---
## 📋 핵심 요약

### Offset 개념

| 용어 | 설명 |
|------|------|
| **Offset** | 파티션 내 메시지의 위치 번호 |
| **Current Offset** | Consumer가 마지막으로 읽은 위치 |
| **Log End Offset** | 파티션의 마지막 메시지 위치 |
| **LAG** | 처리되지 않은 메시지 수 |

### Consumer Group별 Offset

| 특징 | 설명 |
|------|------|
| **독립적** | 각 Group은 자체 Offset 유지 |
| **공유됨** | 같은 Group 내 Consumer는 Offset 공유 |
| **자동 저장** | `__consumer_offsets` 토픽에 저장 |

### Offset 리셋 명령어

| 옵션 | 설명 |
|------|------|
| `--to-earliest` | 처음으로 이동 |
| `--to-latest` | 최신으로 이동 |
| `--to-offset N` | 특정 offset으로 이동 |
| `--shift-by N` | 현재 위치에서 N만큼 이동 |
| `--dry-run` | 미리보기 (실행 안 함) |
| `--execute` | 실제 실행 |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Offset이 무엇인지 설명할 수 있나요? | ☐ |
| 같은 Group vs 다른 Group의 Offset 차이를 이해했나요? | ☐ |
| CLI로 Consumer Group의 Offset을 확인할 수 있나요? | ☐ |
| Offset을 리셋하여 처음부터 다시 읽을 수 있나요? | ☐ |
| LAG가 무엇이고 왜 모니터링해야 하는지 알고 있나요? | ☐ |

---


# Day 07 - 5교시: 멀티 브로커 클러스터 실습
> 여러 브로커로 구성된 Kafka 클러스터에서 Replication과 장애 복구 체험하기

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **멀티 브로커 클러스터**를 Docker Compose로 구성할 수 있다
- **Replication Factor**를 설정하여 토픽을 생성할 수 있다
- **Leader/Replica**의 역할을 이해하고 확인할 수 있다
- 브로커 장애 시 **Leader 재선출**을 관찰할 수 있다

---

## 📚 Day 06 복습: Replication이란?

### 왜 Replication이 필요한가요?

> **Replication = 같은 데이터를 여러 브로커에 복사하는 것**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  단일 브로커 (위험!)                                                     │
│                                                                         │
│   Broker 1 ──────▶ 장애 발생! ──────▶ 데이터 손실!! 💀                   │
│   ┌─────────┐                                                           │
│   │ Topic A │                                                           │
│   │ (유일본) │                                                           │
│   └─────────┘                                                           │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│  멀티 브로커 + Replication (안전!)                                       │
│                                                                         │
│   Broker 1         Broker 2         Broker 3                            │
│   ┌─────────┐     ┌─────────┐     ┌─────────┐                           │
│   │ Topic A │     │ Topic A │     │ Topic A │                           │
│   │ (Leader)│ ◀─▶ │(Replica)│ ◀─▶ │(Replica)│                           │
│   └─────────┘     └─────────┘     └─────────┘                           │
│       ↓                                ↓                                │
│   장애 발생!          Broker 2가 새 Leader로! ✅                         │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Replication Factor

| Factor | 설명 | 권장 상황 |
|--------|------|----------|
| **1** | 복제 없음 | 개발/테스트 환경만 |
| **2** | 1개 백업 | 최소한의 안정성 |
| **3** | 2개 백업 | **프로덕션 표준** |

---
## 🛠️ 실습 준비: 기존 환경 정리

### 1교시 환경 종료

기존 단일 브로커 환경을 종료합니다.

```bash
# ============================================================
# 기존 환경 종료 (kafka-ui-demo 폴더에서)
# ============================================================
cd kafka-ui-demo
docker compose down -v
```

### 새 프로젝트 폴더 생성

```bash
# ============================================================
# 멀티 브로커 실습 폴더 생성
# ============================================================
mkdir kafka-cluster-demo
cd kafka-cluster-demo
```

---
## 🐳 멀티 브로커 Docker Compose 설정

### compose.yml 생성

아래 내용을 `compose.yml` 파일로 저장합니다.

```yaml
# ============================================================
# compose.yml - Kafka 3-Broker Cluster + Kafka UI
# ============================================================

services:
  # ─────────────────────────────────────────────────────────
  # Broker 1 (Controller 역할 겸임)
  # ─────────────────────────────────────────────────────────
  broker-1:
    image: apache/kafka:latest
    container_name: broker-1
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-1:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  # ─────────────────────────────────────────────────────────
  # Broker 2
  # ─────────────────────────────────────────────────────────
  broker-2:
    image: apache/kafka:latest
    container_name: broker-2
    ports:
      - "9093:9092"
    environment:
      KAFKA_NODE_ID: 2
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-2:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  # ─────────────────────────────────────────────────────────
  # Broker 3
  # ─────────────────────────────────────────────────────────
  broker-3:
    image: apache/kafka:latest
    container_name: broker-3
    ports:
      - "9094:9092"
    environment:
      KAFKA_NODE_ID: 3
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-3:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  # ─────────────────────────────────────────────────────────
  # Kafka UI
  # ─────────────────────────────────────────────────────────
  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    depends_on:
      - broker-1
      - broker-2
      - broker-3
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local-cluster
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker-1:9092,broker-2:9092,broker-3:9092
```

### 주요 설정 설명

| 설정 | 값 | 설명 |
|------|-----|------|
| **KAFKA_NODE_ID** | 1, 2, 3 | 각 브로커의 고유 ID |
| **CONTROLLER_QUORUM_VOTERS** | 1@...,2@...,3@... | 클러스터 투표 참여자 목록 |
| **DEFAULT_REPLICATION_FACTOR** | 3 | 토픽 기본 복제 수 |
| **MIN_INSYNC_REPLICAS** | 2 | 쓰기 성공에 필요한 최소 복제본 수 |
| **CLUSTER_ID** | 동일값 | 모든 브로커가 같은 클러스터임을 표시 |

---
## 🚀 클러스터 실행

### Docker Compose 실행

```bash
# ============================================================
# 클러스터 실행
# ============================================================
docker compose up -d
```

### 실행 확인

```bash
# ============================================================
# 컨테이너 상태 확인
# ============================================================
docker ps
```

**예상 출력:**
```
CONTAINER ID   IMAGE                           STATUS         PORTS                    NAMES
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9092->9092/tcp   broker-1
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9093->9092/tcp   broker-2
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9094->9092/tcp   broker-3
xxxxxxxxxxxx   provectuslabs/kafka-ui:latest  Up 25 seconds   0.0.0.0:8080->8080/tcp   kafka-ui
```

> 💡 4개 컨테이너가 모두 실행 중이어야 합니다!

### Kafka UI에서 클러스터 확인

**http://localhost:8080 → Dashboard**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Kafka UI - Dashboard                                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Cluster: local-cluster                                                 │
│                                                                         │
│  ┌──────────────┬──────────────┬──────────────┐                        │
│  │ Brokers: 3   │ Topics: 0    │ Partitions: 0│                        │
│  └──────────────┴──────────────┴──────────────┘                        │
│                                                                         │
│  Brokers:                                                               │
│  ┌──────────┬────────────────┬────────────┐                            │
│  │ Broker ID│ Host           │ Status     │                            │
│  ├──────────┼────────────────┼────────────┤                            │
│  │ 1        │ broker-1:9092  │ Online ✅  │                            │
│  │ 2        │ broker-2:9092  │ Online ✅  │                            │
│  │ 3        │ broker-3:9092  │ Online ✅  │                            │
│  └──────────┴────────────────┴────────────┘                            │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🧪 실습 1: Replication Factor 설정 토픽 생성

### Replication Factor 3으로 토픽 생성

```bash
# ============================================================
# 파티션 3개, Replication Factor 3인 토픽 생성
# ============================================================
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic replicated-topic \
  --partitions 3 \
  --replication-factor 3
```

### 토픽 정보 확인

```bash
# ============================================================
# 토픽 상세 정보 확인
# ============================================================
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   TopicId: xxxxx   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 1   Replicas: 1,2,3   Isr: 1,2,3
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3,1
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,1,2
```

### 출력 해석

| 항목 | 의미 |
|------|------|
| **Leader** | 해당 파티션의 읽기/쓰기를 담당하는 브로커 |
| **Replicas** | 해당 파티션의 복제본을 가진 브로커 목록 |
| **Isr** | 동기화된 복제본 (In-Sync Replicas) |

```
┌─────────────────────────────────────────────────────────────────────────┐
│  파티션 분배 현황                                                        │
│                                                                         │
│   Broker 1              Broker 2              Broker 3                  │
│   ┌─────────────┐      ┌─────────────┐      ┌─────────────┐            │
│   │ P0 (Leader) │      │ P0 (Replica)│      │ P0 (Replica)│            │
│   │ P1 (Replica)│      │ P1 (Leader) │      │ P1 (Replica)│            │
│   │ P2 (Replica)│      │ P2 (Replica)│      │ P2 (Leader) │            │
│   └─────────────┘      └─────────────┘      └─────────────┘            │
│                                                                         │
│   💡 각 파티션은 3개 브로커 모두에 복제됨                                  │
│   💡 Leader가 각 브로커에 분산되어 부하 분배                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🧪 실습 2: 메시지 전송 및 복제 확인

### 메시지 전송

```bash
# ============================================================
# broker-1을 통해 메시지 전송
# ============================================================
docker exec -i broker-1 /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic << 'EOF'
message-1
message-2
message-3
message-4
message-5
EOF
```

### 다른 브로커에서 메시지 읽기

broker-2에서 읽어도 같은 메시지가 보입니다.

```bash
# ============================================================
# broker-2를 통해 메시지 읽기
# ============================================================
docker exec broker-2 /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic \
  --from-beginning
```

**예상 출력:**
```
message-1
message-2
message-3
message-4
message-5
```

> 💡 어떤 브로커에서 읽어도 동일한 메시지!
> 복제가 정상 동작하고 있습니다.

---
## 🧪 실습 3: 브로커 장애 시뮬레이션

### 현재 Leader 확인

```bash
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

예: Partition 0의 Leader가 **브로커 1**이라고 가정

### Leader 브로커 강제 종료

```bash
# ============================================================
# Partition 0의 Leader (broker-1) 종료
# ============================================================
docker stop broker-1
```

### 잠시 대기 후 Leader 재선출 확인

```bash
# ============================================================
# Leader 재선출 확인 (broker-2를 통해)
# ============================================================
sleep 5
docker exec broker-2 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 2   Replicas: 1,2,3   Isr: 2,3
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,2
```

### 변화 분석

| 항목 | 변화 |
|------|------|
| **Leader** | Partition 0의 Leader가 1 → 2로 변경 |
| **Isr** | 브로커 1이 Isr에서 제외됨 (다운 상태) |
| **서비스** | 정상 동작! 메시지 읽기/쓰기 가능 |

```
┌─────────────────────────────────────────────────────────────────────────┐
│  장애 복구 과정                                                          │
│                                                                         │
│   Before (정상):                                                        │
│   ┌───────────┐    ┌───────────┐    ┌───────────┐                      │
│   │ Broker 1  │    │ Broker 2  │    │ Broker 3  │                      │
│   │ P0 Leader │    │ P0 Replica│    │ P0 Replica│                      │
│   └───────────┘    └───────────┘    └───────────┘                      │
│                                                                         │
│   After (broker-1 다운):                                                │
│   ┌───────────┐    ┌───────────┐    ┌───────────┐                      │
│   │ Broker 1  │    │ Broker 2  │    │ Broker 3  │                      │
│   │   ❌ DOWN  │    │ P0 Leader │    │ P0 Replica│                      │
│   └───────────┘    │ (승격!) ✅ │    └───────────┘                      │
│                    └───────────┘                                        │
│                                                                         │
│   💡 Replica 중 하나가 자동으로 Leader로 승격!                            │
│   💡 서비스 중단 없이 계속 동작!                                          │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🧪 실습 4: 장애 상태에서 메시지 처리

### 장애 상태에서 메시지 전송 (broker-2 통해)

```bash
# ============================================================
# broker-1이 다운된 상태에서도 메시지 전송 가능!
# ============================================================
docker exec -i broker-2 /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic << 'EOF'
after-failure-1
after-failure-2
after-failure-3
EOF
```

### 메시지 읽기 확인

```bash
docker exec broker-3 /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic \
  --from-beginning
```

**예상 출력:**
```
message-1
message-2
message-3
message-4
message-5
after-failure-1
after-failure-2
after-failure-3
```

> 브로커 1이 다운됐어도 메시지 읽기/쓰기 정상!

---
## 🧪 실습 5: 브로커 복구 및 동기화

### 다운된 브로커 재시작

```bash
# ============================================================
# broker-1 재시작
# ============================================================
docker start broker-1
```

### 동기화 상태 확인

```bash
# 잠시 대기 후 확인
sleep 10
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 2   Replicas: 1,2,3   Isr: 2,3,1
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3,1
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,2,1
```

### 확인 포인트

| 항목 | 상태 |
|------|------|
| **Isr** | 브로커 1이 다시 Isr에 포함됨 ✅ |
| **Leader** | 이전 Leader로 복구되지 않음 (현재 Leader 유지) |
| **데이터** | 다운 중 전송된 메시지도 자동 동기화됨 |

> 💡 복구된 브로커는 자동으로 누락된 데이터를 동기화합니다!

---
## 🌐 Kafka UI에서 클러스터 상태 확인

### Brokers 메뉴

**http://localhost:8080 → Brokers**

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Kafka UI - Brokers                                                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌──────────┬────────────────┬──────────────┬──────────────┐           │
│  │ Broker ID│ Host           │ Partitions   │ Status       │           │
│  ├──────────┼────────────────┼──────────────┼──────────────┤           │
│  │ 1        │ broker-1:9092  │ 3 (1 leader) │ Online ✅    │           │
│  │ 2        │ broker-2:9092  │ 3 (2 leader) │ Online ✅    │           │
│  │ 3        │ broker-3:9092  │ 3 (0 leader) │ Online ✅    │           │
│  └──────────┴────────────────┴──────────────┴──────────────┘           │
│                                                                         │
│  💡 Leader 파티션이 브로커들에 분산되어 있음                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Topics 상세에서 Replica 확인

**Topics → replicated-topic → Overview**

각 파티션의 Leader와 Replica 상태를 시각적으로 확인할 수 있습니다.

---
## 🧹 실습 정리

### 클러스터 종료

```bash
# ============================================================
# 모든 컨테이너 종료 및 볼륨 삭제
# ============================================================
docker compose down -v
```

### 1교시 환경으로 복구 (선택)

다음 실습을 위해 단일 브로커 환경으로 돌아갑니다.

```bash
cd ../kafka-ui-demo
docker compose up -d
```

---
## ❓ FAQ

**Q1. Replication Factor를 브로커 수보다 크게 할 수 있나요?**

아니요, 불가능합니다. 브로커가 3개면 최대 Replication Factor는 3입니다.
더 큰 값을 지정하면 토픽 생성이 실패합니다.

---

**Q2. Leader가 다운되면 Producer/Consumer는 어떻게 알아요?**

Kafka 클라이언트는 메타데이터를 주기적으로 갱신합니다.
Leader 변경 시 자동으로 새 Leader에 연결합니다.
일시적으로 오류가 발생할 수 있지만, 재시도로 복구됩니다.

---

**Q3. ISR에서 제외된 Replica는 데이터가 유실되나요?**

아니요, 데이터는 보존됩니다.
다만 Leader와 동기화가 늦어진 상태입니다.
브로커가 복구되면 자동으로 동기화되어 ISR에 다시 포함됩니다.

---

**Q4. min.insync.replicas는 무엇인가요?**

Producer가 메시지를 전송할 때 최소 몇 개의 Replica가 동기화되어야 하는지 설정합니다.
- `min.insync.replicas=2`이고 ISR이 1개만 남으면 쓰기 실패
- 데이터 손실을 방지하기 위한 안전장치입니다.

---

**Q5. 프로덕션에서 권장하는 설정은?**

- 브로커: 최소 3대 (홀수 권장)
- Replication Factor: 3
- min.insync.replicas: 2
- acks: all (Producer 설정)

---
## 📝 퀴즈

### Q1. Replication Factor 3으로 토픽을 생성하면?

- A) 메시지가 3배 빨리 전송됨
- B) 각 파티션이 3개 브로커에 복제됨
- C) 파티션이 3개 생성됨
- D) Consumer가 3배 빨리 읽음

<details>
<summary>정답 보기</summary>

**정답: B) 각 파티션이 3개 브로커에 복제됨**

Replication Factor는 각 파티션의 복제본 수를 의미합니다.
3으로 설정하면 각 파티션 데이터가 3개 브로커에 저장됩니다.
</details>

---

### Q2. Leader 브로커가 다운되면 어떻게 되나요?

- A) 서비스가 완전히 중단됨
- B) 데이터가 모두 손실됨
- C) ISR 중 하나가 새 Leader로 선출됨
- D) 관리자가 수동으로 Leader를 지정해야 함

<details>
<summary>정답 보기</summary>

**정답: C) ISR 중 하나가 새 Leader로 선출됨**

Kafka는 자동으로 ISR(동기화된 복제본) 중 하나를 새 Leader로 선출합니다.
서비스 중단 시간을 최소화합니다.
</details>

---

### Q3. ISR(In-Sync Replicas)이란?

- A) 초기 동기화된 Replica
- B) Leader와 동기화가 완료된 Replica 목록
- C) 인터넷에 연결된 Replica
- D) 내부 시스템 Replica

<details>
<summary>정답 보기</summary>

**정답: B) Leader와 동기화가 완료된 Replica 목록**

ISR은 Leader의 최신 데이터를 모두 복제한 Replica들입니다.
Leader 장애 시 ISR 중에서만 새 Leader가 선출됩니다.
</details>

---

### Q4. 브로커가 3개일 때 권장하는 min.insync.replicas 값은?

- A) 1
- B) 2
- C) 3
- D) 4

<details>
<summary>정답 보기</summary>

**정답: B) 2**

브로커 3개, Replication Factor 3일 때 min.insync.replicas=2가 적절합니다.
- 1개 브로커 장애 시에도 쓰기 가능 (ISR 2개 남음)
- 데이터 손실 위험 최소화
</details>

---
## 📋 핵심 요약

### Replication 개념

| 용어 | 설명 |
|------|------|
| **Leader** | 읽기/쓰기를 담당하는 주 복제본 |
| **Replica** | Leader의 데이터를 복제한 복사본 |
| **ISR** | Leader와 동기화된 Replica 목록 |
| **Replication Factor** | 각 파티션의 복제본 수 |

### 장애 복구 흐름

| 단계 | 동작 |
|------|------|
| 1. Leader 다운 | ISR에서 제외됨 |
| 2. Leader 선출 | ISR 중 하나가 새 Leader로 |
| 3. 서비스 계속 | 클라이언트는 새 Leader에 연결 |
| 4. 복구 후 | 자동 동기화, ISR에 재참가 |

### 권장 설정 (프로덕션)

| 설정 | 값 |
|------|-----|
| 브로커 수 | 3개 이상 (홀수) |
| Replication Factor | 3 |
| min.insync.replicas | 2 |
| Producer acks | all |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Docker Compose로 멀티 브로커 클러스터를 실행할 수 있나요? | ☐ |
| Replication Factor를 지정하여 토픽을 생성할 수 있나요? | ☐ |
| Leader와 Replica의 차이를 설명할 수 있나요? | ☐ |
| 브로커 장애 시 Leader 재선출 과정을 이해했나요? | ☐ |
| ISR이 무엇이고 왜 중요한지 알고 있나요? | ☐ |

---


# Day 07 - 6교시: Kafka 설정 커스터마이징
> 환경 변수로 Kafka 설정 변경하기, 주요 설정 항목 이해하기

## 🎯 학습 목표
이 파트를 마치면 다음을 이해할 수 있습니다:

- Kafka의 기본 설정이 어디서 오는지 이해한다
- **환경 변수**로 설정을 변경하는 방법을 안다
- 주요 설정 항목(파티션, 복제, 리스너 등)의 의미를 안다
- QuickStart와 커스텀 설정의 차이를 설명할 수 있다

---

## 📚 1교시 복습: 왜 설정을 배워야 할까?

```
1교시에서 사용한 docker-compose.yml:
─────────────────────────────────────────────────────────────
environment:
  KAFKA_NODE_ID: 1
  KAFKA_PROCESS_ROLES: broker,controller
  KAFKA_LISTENERS: ...
  KAFKA_NUM_PARTITIONS: 3     ← 이런 설정들이 많았음!
  ...

🤔 자연스러운 질문들:
• 이 설정들은 각각 무슨 의미인가?
• 왜 이렇게 많은 환경 변수가 필요한가?
• 어떤 설정을 어떻게 바꿀 수 있는가?

→ 이번 교시에서 답을 찾아봅니다!
```

---
## 🔍 기본 설정은 어디서 오나요?

### Day 06 QuickStart 방식 돌아보기

```bash
# Day 06에서 사용한 간단한 실행 명령
docker run -d --name broker apache/kafka:latest
```

이 한 줄로 Kafka가 "알아서" 돌아갔습니다. 왜?

### 기본 설정이 숨어 있는 곳

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    apache/kafka:latest 이미지 내부                       │
│                                                                         │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │                      기본 설정 파일                              │   │
│   │   /etc/kafka/docker/server.properties                           │   │
│   │   ─────────────────────────────────────                         │   │
│   │   num.partitions=1                    ← 기본 파티션 수            │   │
│   │   default.replication.factor=1        ← 기본 복제 팩터           │   │
│   │   log.retention.hours=168             ← 메시지 보관 시간 (7일)    │   │
│   │   ...                                                           │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│   💡 이 설정들이 "알아서" 적용되어 Kafka가 바로 실행된 것!                 │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 실습: 기본값 확인하기

```bash
# ============================================================
# 기본 설정으로 Kafka 실행 후 파티션 수 확인
# ============================================================

# 1. Kafka 실행 (QuickStart 방식)
docker run -d --name broker apache/kafka:latest

# 2. 토픽 생성 (파티션 수 지정 안 함)
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic check-default

# 3. 토픽 정보 확인
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic check-default
```

**예상 출력:**
```
Topic: check-default    PartitionCount: 1    ReplicationFactor: 1
    Topic: check-default    Partition: 0    Leader: 1    ...
```

**파티션이 1개!** 이것이 기본 설정(`num.partitions=1`)의 결과입니다.

```bash
# 정리
docker rm -f broker
```

---
## ⚙️ 환경 변수로 설정 변경하기

### 왜 환경 변수를 사용하나요?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  설정을 변경하는 방법들:                                                  │
│                                                                         │
│  방법 1: 설정 파일 직접 수정                                              │
│  ─────────────────────────                                              │
│    • 컨테이너 안에 들어가서 파일 편집                                      │
│    • 번거롭고, 컨테이너 재시작하면 사라질 수 있음                           │
│    • ❌ 비추천                                                           │
│                                                                         │
│  방법 2: 환경 변수 사용                                                   │
│  ─────────────────────────                                              │
│    • docker run -e 또는 docker-compose.yml의 environment                │
│    • 컨테이너 외부에서 설정을 주입                                         │
│    • 버전 관리(Git) 가능, 재현 가능                                       │
│    • ✅ 추천!                                                            │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### ⚠️ 중요한 규칙: "전부 아니면 전무"

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│   ⚠️ Apache Kafka Docker 이미지의 특성:                                  │
│                                                                         │
│   환경 변수를 하나라도 지정하면 → 기본 설정 파일이 무시됨!                  │
│                                                                         │
│   즉, 환경 변수로 설정을 시작하면                                         │
│   필요한 모든 설정을 직접 명시해야 합니다.                                 │
│                                                                         │
│   이것이 docker-compose.yml에 환경 변수가 많은 이유입니다!                 │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 환경 변수 네이밍 규칙

```
설정 파일 (server.properties)    →    환경 변수
─────────────────────────────────────────────────────────
num.partitions=3                 →    KAFKA_NUM_PARTITIONS=3
log.retention.hours=168          →    KAFKA_LOG_RETENTION_HOURS=168
auto.create.topics.enable=true   →    KAFKA_AUTO_CREATE_TOPICS_ENABLE=true

규칙:
1. 앞에 KAFKA_ 접두사 추가
2. 점(.)을 언더스코어(_)로 변환
3. 대문자로 변환
```

---
## 📋 주요 설정 항목 이해하기

### 설정 카테고리 개요

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Kafka 설정 카테고리                               │
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  🆔 노드 식별                                                    │   │
│  │  KAFKA_NODE_ID: 1                                               │   │
│  │  "나는 1번 브로커야"                                              │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                         │                                              │
│                         ▼                                              │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  🎭 역할 지정 (KRaft 모드)                                        │   │
│  │  KAFKA_PROCESS_ROLES: broker,controller                         │   │
│  │  "나는 브로커도 하고 컨트롤러도 할게"                               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                         │                                              │
│                         ▼                                              │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  🌐 네트워크 설정                                                │   │
│  │  KAFKA_LISTENERS: 어디서 연결 받을지                              │   │
│  │  KAFKA_ADVERTISED_LISTENERS: 클라이언트에게 알려줄 주소            │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                         │                                              │
│                         ▼                                              │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  ⚙️ 동작 설정                                                    │   │
│  │  KAFKA_NUM_PARTITIONS: 3  ← 기본 파티션 수 변경!                  │   │
│  │  "토픽 만들 때 기본으로 3개 파티션"                                 │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 1. 노드 식별 설정

```yaml
# ═══════════════════════════════════════════════════════════════
# 🆔 노드 식별 설정
# ═══════════════════════════════════════════════════════════════
KAFKA_NODE_ID: 1
```

| 설정 | 설명 |
|------|------|
| **역할** | 이 브로커의 고유 ID |
| **값** | 정수 (1, 2, 3, ...) |
| **중요** | 클러스터 내에서 중복되면 안 됨 |

```
단일 노드:     멀티 노드 클러스터:
┌─────────┐    ┌─────────┐ ┌─────────┐ ┌─────────┐
│ NODE_ID │    │ NODE_ID │ │ NODE_ID │ │ NODE_ID │
│    1    │    │    1    │ │    2    │ │    3    │
└─────────┘    └─────────┘ └─────────┘ └─────────┘
```

### 2. 역할 설정

```yaml
# ═══════════════════════════════════════════════════════════════
# 🎭 역할 설정
# ═══════════════════════════════════════════════════════════════
KAFKA_PROCESS_ROLES: broker,controller
```

Kafka 노드는 **브로커**(메시지 저장/전달)와 **컨트롤러**(클러스터 관리) 역할을 수행합니다.

| 값 | 설명 | 사용 상황 |
|-----|------|----------|
| `broker,controller` | 두 역할 모두 수행 | 개발/학습 환경 (**우리가 사용**) |
| `broker` | 메시지 저장/전달만 | 프로덕션 (역할 분리) |
| `controller` | 클러스터 관리만 | 프로덕션 (역할 분리) |

> 💡 개발 환경에서는 단일 노드가 두 역할을 모두 수행합니다.
> 프로덕션에서는 안정성을 위해 역할을 분리하는 것이 일반적입니다.

### 3. 네트워크/리스너 설정 (가장 혼란스러운 부분!)

```yaml
# ═══════════════════════════════════════════════════════════════
# 🌐 네트워크/리스너 설정
# ═══════════════════════════════════════════════════════════════
KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
```

#### LISTENERS vs ADVERTISED_LISTENERS

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  KAFKA_LISTENERS: "내가 어디서 연결을 받을지"                              │
│  ─────────────────────────────────────────                              │
│  PLAINTEXT://0.0.0.0:9092                                               │
│  • 0.0.0.0 = 모든 네트워크 인터페이스에서 수신                             │
│  • 9092 = Kafka 기본 포트                                                │
│  • PLAINTEXT = 암호화 없는 일반 연결                                      │
│                                                                         │
│  KAFKA_ADVERTISED_LISTENERS: "클라이언트에게 알려줄 내 주소"               │
│  ────────────────────────────────────────────                           │
│  PLAINTEXT://broker:9092                                                │
│  • broker = Docker 컨테이너 이름 (DNS로 등록됨)                           │
│  • 클라이언트가 "이 주소로 나한테 연결해"라고 안내받는 주소                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

#### 왜 두 개가 다를 수 있나요?

```
예시: Docker 환경
─────────────────────────────────────────────────────────

┌────────────────────────────────────────────────────────┐
│  Docker 네트워크                                         │
│                                                        │
│   ┌──────────────────┐      ┌──────────────────┐       │
│   │  kafka-ui        │ ────▶│  broker          │       │
│   │                  │      │                  │       │
│   │  ADVERTISED를    │      │  LISTENERS로      │       │
│   │  보고 연결 시도     │      │  연결 수신         │        │
│   │  "broker:9092"   │      │  "0.0.0.0:9092"  │       │
│   └──────────────────┘      └──────────────────┘       │
│                                                        │
└────────────────────────────────────────────────────────┘

LISTENERS: 0.0.0.0:9092 → "모든 곳에서 오는 연결 받아들임"
ADVERTISED: broker:9092 → "나한테 연결하려면 broker:9092로 와"
```

### 4. 컨트롤러 설정

```yaml
# ═══════════════════════════════════════════════════════════════
# 🔧 컨트롤러 설정
# ═══════════════════════════════════════════════════════════════
KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
```

| 설정 | 설명 |
|------|------|
| **형식** | `노드ID@호스트:포트` |
| **역할** | 컨트롤러 투표 그룹 구성 |
| **단일 노드** | `1@broker:9093` |
| **멀티 노드** | `1@broker1:9093,2@broker2:9093,3@broker3:9093` |

```
단일 노드:                    멀티 노드 (3개):
1@broker:9093                1@broker1:9093,
                             2@broker2:9093,
                             3@broker3:9093

컨트롤러 선출에 참여하는 노드 목록
```

### 5. 내부 토픽 설정

```yaml
# ═══════════════════════════════════════════════════════════════
# 📦 내부 토픽 설정 (단일 노드 환경용)
# ═══════════════════════════════════════════════════════════════
KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
```

| 설정 | 의미 | 단일 노드 | 프로덕션 권장 |
|------|------|----------|-------------|
| `OFFSETS_TOPIC_REPLICATION_FACTOR` | Consumer 오프셋 저장 토픽의 복제 수 | 1 | 3 |
| `TRANSACTION_STATE_LOG_REPLICATION_FACTOR` | 트랜잭션 상태 토픽의 복제 수 | 1 | 3 |
| `TRANSACTION_STATE_LOG_MIN_ISR` | 최소 동기화 복제본 수 | 1 | 2 |

> ⚠️ **주의**: 단일 브로커에서는 복제가 불가능하므로 모두 1로 설정해야 합니다.
> 멀티 노드 클러스터에서는 3 이상을 권장합니다.

### 6. 동작/성능 설정 ← 주로 변경하는 부분!

```yaml
# ═══════════════════════════════════════════════════════════════
# ⚙️ 동작/성능 설정
# ═══════════════════════════════════════════════════════════════
KAFKA_NUM_PARTITIONS: 3
KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
```

#### 자주 변경하는 설정들

| 설정 | 기본값 | 설명 | 변경 예시 |
|------|--------|------|----------|
| `NUM_PARTITIONS` | 1 | 토픽 기본 파티션 수 | 3, 6, 12 |
| `LOG_RETENTION_HOURS` | 168 (7일) | 메시지 보관 시간 | 24, 72, 720 |
| `LOG_RETENTION_BYTES` | -1 (무제한) | 토픽당 최대 용량 | 1073741824 (1GB) |
| `MESSAGE_MAX_BYTES` | 1048588 (~1MB) | 최대 메시지 크기 | 10485880 (10MB) |
| `AUTO_CREATE_TOPICS_ENABLE` | true | 자동 토픽 생성 | false (프로덕션) |

```
NUM_PARTITIONS 변경 효과:
─────────────────────────────────────────────────────────

기본값 (1개):               변경 후 (3개):
┌─────────────────┐        ┌─────────────────┐
│  Partition 0    │        │  Partition 0    │
│  (모든 메시지)     │        │  Partition 1    │
└─────────────────┘        │  Partition 2    │
                           └─────────────────┘

• 파티션 1개: 순차 처리만 가능
• 파티션 3개: 최대 3개의 Consumer가 병렬 처리 가능!
```

---
## 🧪 실습: 설정 변경 확인하기

### Step 1: docker-compose.yml 작성

```yaml
# ============================================================
# docker-compose.yml - 파티션 3개로 설정
# ============================================================

services:
  broker:
    image: apache/kafka:latest
    container_name: broker
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      # ─────────────────────────────────────────────
      # ⭐ 핵심 설정: 기본 파티션 수를 3으로!
      # ─────────────────────────────────────────────
      KAFKA_NUM_PARTITIONS: 3

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    depends_on:
      - broker
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local-kafka
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker:9092
```

### Step 2: 실행 및 확인

```bash
# ============================================================
# 1. 실행
# ============================================================
docker compose up -d

# ============================================================
# 2. 토픽 생성 (파티션 수 지정 안 함!)
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic auto-partition-topic

# ============================================================
# 3. 토픽 정보 확인
# ============================================================
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic auto-partition-topic
```

### 예상 출력

```
Topic: auto-partition-topic   PartitionCount: 3   ReplicationFactor: 1
    Topic: auto-partition-topic   Partition: 0    Leader: 1   ...
    Topic: auto-partition-topic   Partition: 1    Leader: 1   ...
    Topic: auto-partition-topic   Partition: 2    Leader: 1   ...
```

**파티션이 3개로 생성되었습니다!** `KAFKA_NUM_PARTITIONS: 3` 설정이 적용된 것입니다.

### 설정 변경 전후 비교

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  Day 06 (QuickStart)                  Day 07 (커스텀 설정)               │
│  ─────────────────────                ─────────────────────             │
│                                                                         │
│  기본 설정 사용                        환경 변수로 설정 지정                │
│  num.partitions = 1                   KAFKA_NUM_PARTITIONS = 3          │
│                                                                         │
│  토픽: test-topic                     토픽: auto-partition-topic         │
│  ┌─────────────────┐                 ┌─────────────────┐                │
│  │  Partition 0    │                 │  Partition 0    │                │
│  │                 │                 │  Partition 1    │                │
│  └─────────────────┘                 │  Partition 2    │                │
│                                      └─────────────────┘                │
│                                                                         │
│  파티션 1개 → 처리량 제한             파티션 3개 → 병렬 처리 가능           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🔧 토픽 생성 시 설정 직접 지정하기

기본값과 다르게 토픽을 생성할 수도 있습니다.

### 파티션 수 직접 지정

```bash
# ============================================================
# 기본값(3개)과 다르게 10개 파티션으로 생성
# ============================================================

docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic custom-partitions \
  --partitions 10
```

**결과**: 이 토픽은 파티션 10개로 생성됩니다 (기본값 무시)

### 토픽 레벨 설정 추가

```bash
# ============================================================
# 토픽 생성 시 보관 기간도 설정
# ============================================================

docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic short-retention \
  --partitions 3 \
  --config retention.ms=86400000    # 1일 (밀리초)
```

### 설정 우선순위

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         설정 우선순위                                      │
│                                                                         │
│   낮음 ◀────────────────────────────────────────────────────▶ 높음        │
│                                                                         │
│   ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐        │
│   │  기본 설정        │  │  브로커 설정       │  │  토픽별 설정       │        │
│   │  (이미지 내장)     │  │  (환경 변수)      │  │  (--config)     │        │
│   └─────────────────┘  └─────────────────┘  └─────────────────┘        │
│                                                                         │
│   예: num.partitions     KAFKA_NUM_PARTITIONS    --partitions 10         │
│       = 1                = 3                      = 10                  │
│                                                                         │
│   → 토픽별 설정이 있으면 그게 적용됨!                                           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## ❓ FAQ

**Q1. 파티션 수는 몇 개가 적당한가요?**

정답은 없지만 일반적인 가이드라인:
- 개발/테스트: 1~3개
- 소규모 프로덕션: 6~12개

파티션이 많으면 병렬 처리는 좋지만 관리 오버헤드가 증가합니다.

---

**Q2. 이미 생성된 토픽의 파티션 수를 바꿀 수 있나요?**

늘릴 수는 있지만, 줄일 수는 없습니다:
```bash
# 파티션 증가 (3 → 6)
kafka-topics.sh --alter --topic my-topic --partitions 6
```

⚠️ 단, 파티션 증가는 메시지 순서에 영향을 줄 수 있으니 주의하세요.

---

**Q3. ADVERTISED_LISTENERS를 잘못 설정하면 어떻게 되나요?**

클라이언트가 연결 정보를 잘못 받아서 연결에 실패합니다.
특히 Docker 환경에서 흔히 발생하는 오류입니다.

증상: "Connection refused" 또는 "Unknown host"

---

**Q4. 설정 변경 후 재시작이 필요한가요?**

환경 변수로 설정하는 경우, 컨테이너 재시작이 필요합니다:
```bash
docker compose down && docker compose up -d
```

일부 설정(토픽별 설정)은 동적으로 변경 가능합니다.

---
## 📝 퀴즈

### Q1. Kafka 환경 변수로 `num.partitions` 설정을 변경하려면?

- A) KAFKA_PARTITIONS=3
- B) KAFKA_NUM_PARTITIONS=3
- C) NUM_PARTITIONS=3
- D) KAFKA_PARTITION_COUNT=3

<details>
<summary>정답 보기</summary>

**정답: B) KAFKA_NUM_PARTITIONS=3**

규칙: `KAFKA_` 접두사 + 점(.)을 언더스코어(_)로 + 대문자
`num.partitions` → `KAFKA_NUM_PARTITIONS`
</details>

---

### Q2. Apache Kafka Docker 이미지에서 환경 변수를 하나라도 지정하면?

- A) 해당 설정만 변경되고 나머지는 기본값 사용
- B) 기본 설정 파일이 무시되어 필요한 모든 설정을 명시해야 함
- C) 오류가 발생하여 실행되지 않음
- D) 환경 변수와 기본값이 합쳐져서 적용됨

<details>
<summary>정답 보기</summary>

**정답: B) 기본 설정 파일이 무시되어 필요한 모든 설정을 명시해야 함**

"전부 아니면 전무" 규칙입니다.
환경 변수를 하나라도 지정하면 기본 설정 파일이 무시되므로,
필요한 모든 설정을 환경 변수로 지정해야 합니다.
</details>

---

### Q3. KAFKA_LISTENERS와 KAFKA_ADVERTISED_LISTENERS의 차이점은?

- A) LISTENERS는 내부용, ADVERTISED는 외부용
- B) LISTENERS는 수신 주소, ADVERTISED는 클라이언트에게 알려줄 주소
- C) LISTENERS는 TCP, ADVERTISED는 HTTP
- D) 둘은 같은 역할이다

<details>
<summary>정답 보기</summary>

**정답: B) LISTENERS는 수신 주소, ADVERTISED는 클라이언트에게 알려줄 주소**

LISTENERS: "내가 어디서 연결을 받을지" (서버 바인딩 주소)
ADVERTISED: "클라이언트에게 알려줄 내 주소" (클라이언트가 연결할 주소)
</details>

---

### Q4. 토픽 생성 시 파티션 수를 직접 지정하면?

- A) 브로커 기본 설정과 합산됨
- B) 오류 발생
- C) 지정한 값이 적용됨 (기본값 무시)
- D) 둘 중 큰 값이 적용됨

<details>
<summary>정답 보기</summary>

**정답: C) 지정한 값이 적용됨 (기본값 무시)**

토픽별 설정이 가장 높은 우선순위를 가집니다.
`--partitions 10`으로 지정하면 기본값(3)이 아닌 10개가 생성됩니다.
</details>

---

### Q5. 단일 노드 환경에서 OFFSETS_TOPIC_REPLICATION_FACTOR를 3으로 설정하면?

- A) 정상 작동
- B) 오류 발생 (복제 불가)
- C) 자동으로 1로 조정됨
- D) 3개의 가상 복제본 생성

<details>
<summary>정답 보기</summary>

**정답: B) 오류 발생 (복제 불가)**

복제는 여러 브로커에 데이터를 복사하는 것입니다.
단일 브로커에서는 복제가 불가능하므로 반드시 1로 설정해야 합니다.
</details>

---
## 📋 과제

### 과제 1: 파티션 수 변경하기 (난이도: ⭐)

1. docker-compose.yml에서 `KAFKA_NUM_PARTITIONS`를 5로 변경하세요
2. `docker compose down && docker compose up -d`로 재시작
3. 새 토픽을 생성하고 파티션이 5개인지 확인하세요

<details>
<summary>💡 힌트</summary>

```yaml
environment:
  KAFKA_NUM_PARTITIONS: 5    # 3에서 5로 변경
```

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic five-partition-topic

docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe --topic five-partition-topic
```
</details>

---

### 과제 2: 토픽 생성 시 파티션 직접 지정하기 (난이도: ⭐⭐)

기본값(5개)과 다르게 토픽을 생성해보세요:

1. 파티션 1개짜리 토픽 `single-partition` 생성
2. 파티션 10개짜리 토픽 `ten-partitions` 생성
3. 두 토픽을 `--describe`로 확인

**질문**: 기본값(5개)이 적용된 토픽은 어떤 것인가요?

---

### 과제 3: 메시지 보관 기간 설정하기 (난이도: ⭐⭐⭐)

1. 아래 설정을 docker-compose.yml에 추가하세요:
   ```yaml
   KAFKA_LOG_RETENTION_HOURS: 1    # 1시간 후 삭제
   ```
2. 새 토픽을 생성하고 메시지를 보내세요
3. UI에서 토픽 설정을 확인하고, `retention.ms` 값이 얼마인지 확인하세요

> 💡 **힌트**: Kafka UI > Topics > 토픽 선택 > Settings 탭에서 확인 가능

---
## 📋 핵심 요약

### 설정 변경 방법

| 방법 | 예시 | 추천 |
|------|------|------|
| 기본값 사용 | `docker run apache/kafka` | 테스트용 |
| 환경 변수 | `KAFKA_NUM_PARTITIONS: 3` | ✅ 프로덕션 |
| 토픽별 설정 | `--config retention.ms=...` | 특정 토픽만 |

### 환경 변수 네이밍 규칙

```
설정 파일                    환경 변수
──────────────────────────────────────────
num.partitions         →    KAFKA_NUM_PARTITIONS
log.retention.hours    →    KAFKA_LOG_RETENTION_HOURS

규칙: KAFKA_ + 점(.)→언더스코어(_) + 대문자
```

### 주요 설정 카테고리

| 카테고리 | 설정 예시 |
|----------|----------|
| 노드 식별 | `KAFKA_NODE_ID` |
| 역할 지정 | `KAFKA_PROCESS_ROLES` |
| 네트워크 | `KAFKA_LISTENERS`, `KAFKA_ADVERTISED_LISTENERS` |
| 내부 토픽 | `KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR` |
| 동작 설정 | `KAFKA_NUM_PARTITIONS`, `KAFKA_LOG_RETENTION_HOURS` |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| 환경 변수로 Kafka 설정을 변경할 수 있나요? | ☐ |
| 환경 변수 네이밍 규칙을 알고 있나요? | ☐ |
| LISTENERS와 ADVERTISED_LISTENERS의 차이를 설명할 수 있나요? | ☐ |
| NUM_PARTITIONS 설정의 효과를 이해했나요? | ☐ |

---


# Day 07 - 7교시: confluent-kafka로 비동기 시스템 구축하기
> Python에서 Kafka를 활용하여 동기식 시스템을 비동기식으로 전환하는 실습

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **confluent-kafka-python**으로 Producer/Consumer를 구현할 수 있다
- **jq**를 활용하여 JSON 응답을 깔끔하게 처리할 수 있다
- 동기식 → 비동기식 전환의 효과를 직접 체험하고 설명할 수 있다
- Kafka 도입의 장점과 한계를 이해한다

---

## 📚 전체 학습 흐름

| 순서 | 내용 | 시간 |
|------|------|------|
| 1 | 포트 매핑 복습 & jq 도구 소개 | 10분 |
| 2 | confluent-kafka-python 소개 | 15분 |
| 3 | Producer/Consumer 기본 문법 | 20분 |
| 4 | 동기식 → 비동기식 전환 실습 | 40분 |
| 5 | 비교 실습 및 정리 | 20분 |

---
## 1. 포트 매핑 복습

### 왜 포트 매핑이 필요한가?

Docker 컨테이너는 호스트와 **별도의 네트워크 공간**을 가짐:
- 호스트의 `localhost:9092` ≠ 컨테이너의 `localhost:9092`
- 포트 매핑 없이는 호스트에서 컨테이너 내부 서비스에 접근 불가

### 포트 매핑 동작 원리

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  포트 매핑: -p 9092:9092                                                 │
│  ─────────────────────────────                                          │
│                                                                         │
│   호스트                           컨테이너                              │
│   ┌──────────────┐                ┌──────────────┐                      │
│   │              │      ✅        │              │                      │
│   │  :9092 ──────┼───────────────▶│  :9092 Kafka │                      │
│   │  (열림)       │   포트 포워딩     │              │                      │
│   └──────────────┘                └──────────────┘                      │
│                                                                         │
│   이제 호스트에서 localhost:9092 → 컨테이너 Kafka에 도달!                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### compose.yml에서의 포트 매핑

```yaml
services:
  broker:
    image: apache/kafka:latest
    ports:
      - "9092:9092"    # 호스트:컨테이너
```

> 💡 **핵심**: 포트 매핑 덕분에 호스트의 Python 애플리케이션이 컨테이너의 Kafka에 연결 가능!

---
## 2. jq 설치 및 사용법

### jq란?

> **jq**: 커맨드라인에서 JSON 데이터를 파싱하고 가공하는 도구

- JSON을 예쁘게 포맷팅
- 특정 필드만 추출
- **유니코드(한글) 깨짐 문제 해결**

> 📖 **공식 문서**: [jq Manual](https://stedolan.github.io/jq/manual/) 참고

### WSL/Ubuntu에서 설치

```bash
# jq 설치
sudo apt-get update && sudo apt-get install -y jq

# 설치 확인
jq --version
```

### 기본 사용법

| 명령어 | 설명 | 예시 |
|--------|------|------|
| `jq .` | JSON 전체를 예쁘게 출력 | `echo '{"a":1}' \| jq .` |
| `jq '.필드명'` | 특정 필드 추출 | `echo '{"name":"홍길동"}' \| jq '.name'` |
| `jq -r` | 따옴표 없이 raw 출력 (유니코드 정상 출력) | `echo '{"name":"홍길동"}' \| jq -r '.name'` |
| `jq '.[]'` | 배열의 각 요소 출력 | `echo '[1,2,3]' \| jq '.[]'` |

### 유니코드 깨짐 해결

**문제 상황**: curl로 API 호출 시 한글이 `\uD55C\uAE00` 형태로 출력

```bash
# 유니코드가 깨진 출력
curl -s http://localhost:5000/order
# 출력: {"message": "\uc8fc\ubb38\uc774 \uc811\uc218\ub418\uc5c8\uc2b5\ub2c8\ub2e4"}

# jq로 해결 (예쁘게 포맷팅)
curl -s http://localhost:5000/order | jq .
# 출력:
# {
#   "message": "주문이 접수되었습니다"
# }

# 특정 필드만 추출 (-r: 따옴표 제거)
curl -s http://localhost:5000/order | jq -r '.message'
# 출력: 주문이 접수되었습니다
```

---
## 3. confluent-kafka-python 소개

### 왜 confluent-kafka인가?

| 라이브러리 | 특징 | 성능 |
|-----------|------|------|
| kafka-python | 순수 Python, 설치 쉬움 | 느림 |
| **confluent-kafka** | librdkafka 기반 (C 라이브러리) | **빠름** (10배 이상) |
| aiokafka | 비동기 (asyncio) 지원 | 중간 |

**confluent-kafka 선택 이유**:
- Confluent 공식 지원 (Kafka 창시자 회사)
- 프로덕션 검증된 성능
- 풍부한 기능 (트랜잭션, 정확히 한 번 전송 등)

> 📖 **공식 문서**: [confluent-kafka-python](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html) 참고

### 아키텍처 개요

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Kafka 기반 비동기 아키텍처                          │
│                                                                         │
│   ┌──────────┐         ┌──────────────────┐         ┌──────────────┐   │
│   │  Order   │         │      Kafka       │         │  Consumers   │   │
│   │ Service  │         │                  │         │              │   │
│   │          │  발행    │  ┌────────────┐  │  구독    │ [Inventory]  │   │
│   │ Producer │───────▶ │  │   orders   │  │  ◀──────│ [Shipping]   │   │
│   │          │         │  │   topic    │  │         │ [Notification]│   │
│   └──────────┘         │  └────────────┘  │         └──────────────┘   │
│        │               └──────────────────┘               │            │
│        │                                                  │            │
│   즉시 응답!                                         독립적으로 처리      │
│   (0.05초)                                          (각자 속도대로)      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 설치 방법

```bash
# pip으로 설치
pip install confluent-kafka

# 또는 Docker 이미지에 포함 (Dockerfile에서)
RUN pip install confluent-kafka
```

---
## 4. Producer 기본 문법

### Producer 설정 옵션

| 설정 | 설명 | 권장값 |
|------|------|--------|
| `bootstrap.servers` | Kafka 브로커 주소 | `localhost:9092` |
| `client.id` | 클라이언트 식별자 (디버깅용) | `my-producer` |
| `acks` | 메시지 전송 확인 수준 | `all` (가장 안전) |
| `retries` | 전송 실패 시 재시도 횟수 | `3` |
| `linger.ms` | 배치 전송 대기 시간 | `0` (즉시 전송) |

### Producer 기본 코드

In [ ]:
# confluent-kafka Producer 기본 예제
from confluent_kafka import Producer
import json

# ------------------------------------------------------------
# 1. Producer 설정
# ------------------------------------------------------------
config = {
    "bootstrap.servers": "localhost:9092",  # Kafka 브로커 주소
    "client.id": "my-producer",  # 클라이언트 식별자
    "acks": "all",  # 모든 ISR이 복제 완료해야 성공
}

# Producer 인스턴스 생성
producer = Producer(config)


# ------------------------------------------------------------
# 2. 전송 완료 콜백 함수
# ------------------------------------------------------------
def delivery_callback(err, msg):
    """메시지 전송 결과를 처리하는 콜백 함수"""
    if err is not None:
        print(f"❌ 전송 실패: {err}")
    else:
        print(f"✅ 전송 성공: {msg.topic()} [{msg.partition()}] @ {msg.offset()}")


# ------------------------------------------------------------
# 3. 메시지 전송
# ------------------------------------------------------------
order_data = {
    "order_id": "abc123",
    "product": "노트북",
    "quantity": 1,
    "customer": "홍길동",
}

# produce(): 메시지를 전송 큐에 추가 (비동기)
producer.produce(
    topic="orders",
    key=order_data["order_id"].encode("utf-8"),  # 같은 키 → 같은 파티션
    value=json.dumps(order_data).encode("utf-8"),  # JSON → bytes
    callback=delivery_callback,
)

# ------------------------------------------------------------
# 4. 전송 완료 대기
# ------------------------------------------------------------
# flush(): 큐의 모든 메시지가 전송될 때까지 대기
producer.flush(timeout=10)

### Producer 주요 메서드

| 메서드 | 설명 |
|--------|------|
| `produce(topic, value, key, callback)` | 메시지를 전송 큐에 추가 (비동기) |
| `poll(timeout)` | 콜백 함수 처리, timeout=0이면 즉시 반환 |
| `flush(timeout)` | 큐의 모든 메시지 전송 완료 대기 |

---
## 5. Consumer 기본 문법

### Consumer 설정 옵션

| 설정 | 설명 | 권장값 |
|------|------|--------|
| `bootstrap.servers` | Kafka 브로커 주소 | `localhost:9092` |
| `group.id` | Consumer Group ID (필수!) | 서비스별로 다르게 |
| `auto.offset.reset` | 오프셋 없을 때 시작 위치 | `earliest` 또는 `latest` |
| `enable.auto.commit` | 오프셋 자동 커밋 여부 | `True` |
| `session.timeout.ms` | Consumer 죽음 판단 시간 | `30000` (30초) |

### Consumer 기본 코드

In [ ]:
# confluent-kafka Consumer 기본 예제
from confluent_kafka import Consumer, KafkaError
import json

# ------------------------------------------------------------
# 1. Consumer 설정
# ------------------------------------------------------------
config = {
    "bootstrap.servers": "localhost:9092",
    "group.id": "order-processors",  # Consumer Group ID (필수)
    "auto.offset.reset": "earliest",  # 처음부터 읽기
    "enable.auto.commit": True,
    "auto.commit.interval.ms": 5000,  # 5초마다 커밋
    "session.timeout.ms": 30000,
}

# Consumer 인스턴스 생성
consumer = Consumer(config)

# ------------------------------------------------------------
# 2. 토픽 구독
# ------------------------------------------------------------
consumer.subscribe(["orders"])

# ------------------------------------------------------------
# 3. 메시지 폴링
# ------------------------------------------------------------
try:
    while True:
        # poll(): 메시지 하나를 가져옴
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue

        # 에러 체크
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                print(f"파티션 끝 도달: {msg.topic()}[{msg.partition()}]")
            else:
                print(f"에러 발생: {msg.error()}")
            continue

        # 메시지 처리
        key = msg.key().decode("utf-8") if msg.key() else None
        value = json.loads(msg.value().decode("utf-8"))

        print(f"📨 메시지 수신:")
        print(f"   토픽: {msg.topic()}")
        print(f"   파티션: {msg.partition()}")
        print(f"   오프셋: {msg.offset()}")
        print(f"   값: {value}")

except KeyboardInterrupt:
    print("종료 요청...")

finally:
    # ------------------------------------------------------------
    # 4. 정리
    # ------------------------------------------------------------
    consumer.close()

### Consumer 주요 메서드

| 메서드 | 설명 |
|--------|------|
| `subscribe(topics)` | 토픽 구독 (리스트로 여러 개 가능) |
| `poll(timeout)` | 메시지 하나를 가져옴, timeout 동안 대기 |
| `commit()` | 현재까지 처리한 오프셋을 수동 커밋 |
| `close()` | Consumer 종료 및 리소스 정리 |

---
## 6. 동기식 → 비동기식 전환 실습

### 6.1 폴더 구조 준비

```bash
mkdir -p kafka-migration/sync-version/{order,inventory,shipping,notification}
mkdir -p kafka-migration/async-version/{order,inventory,shipping,notification}
cd kafka-migration
```

**최종 구조**:
```
kafka-migration/
├── sync-version/          # 동기식 버전 (기존)
│   ├── compose.yml
│   ├── order/
│   ├── inventory/
│   ├── shipping/
│   └── notification/
│
└── async-version/         # 비동기식 버전 (Kafka)
    ├── compose.yml
    ├── order/
    ├── inventory/
    ├── shipping/
    └── notification/
```

### 6.2 동기식 버전 (문제 상황 체험)

**동기식의 문제점**:
- 모든 서비스를 순차적으로 호출 (직렬 처리)
- 하나라도 느리면 전체가 느려짐
- 하나라도 실패하면 전체 실패

#### sync-version/order/app.py

```python
# ============================================================
# 동기식 주문 서비스
# 문제점: 모든 서비스 순차 호출 → 느리고 장애 전파
# ============================================================
from flask import Flask, jsonify
import requests
import time
import uuid

app = Flask(__name__)

@app.route('/order', methods=['POST'])
def create_order():
    start = time.time()
    order_id = str(uuid.uuid4())[:8]

    print(f"\n{'='*50}")
    print(f"🛒 [동기식] 주문 접수: {order_id}")
    print(f"{'='*50}")

    # Step 1: 재고 확인 (0.5초)
    print("📞 [1/3] 재고 서비스 호출 중...")
    try:
        response = requests.get(
            "http://inventory:5001/check",
            params={"order_id": order_id},
            timeout=5
        )
        print(f"   ✅ 재고 확인 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 재고 서비스 실패: {e}")
        return jsonify({"error": "재고 서비스 연결 실패"}), 500

    # Step 2: 배송 예약 (3초) - 느림!
    print("📞 [2/3] 배송 서비스 호출 중... (오래 걸림)")
    try:
        response = requests.get(
            "http://shipping:5002/schedule",
            params={"order_id": order_id},
            timeout=10
        )
        print(f"   ✅ 배송 예약 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 배송 서비스 실패: {e}")
        return jsonify({"error": "배송 서비스 연결 실패"}), 500

    # Step 3: 알림 발송 (0.3초)
    print("📞 [3/3] 알림 서비스 호출 중...")
    try:
        response = requests.get(
            "http://notification:5003/send",
            params={"order_id": order_id},
            timeout=5
        )
        print(f"   ✅ 알림 발송 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 알림 서비스 실패: {e}")
        return jsonify({"error": "알림 서비스 연결 실패"}), 500

    elapsed = time.time() - start
    print(f"\n⏱️ 총 처리 시간: {elapsed:.2f}초")

    return jsonify({
        "status": "completed",
        "order_id": order_id,
        "elapsed_seconds": round(elapsed, 2),
        "message": f"주문 완료! (처리 시간: {elapsed:.1f}초)"
    })

if __name__ == '__main__':
    print("🚀 [동기식] 주문 서비스 시작 (포트: 5000)")
    app.run(host='0.0.0.0', port=5000)
```

#### sync-version/inventory/app.py

```python
# 재고 서비스 (처리 시간: 0.5초)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/check')
def check_inventory():
    order_id = request.args.get('order_id', 'unknown')
    print(f"📦 재고 확인 중... (주문: {order_id})")
    time.sleep(0.5)  # DB 조회 시뮬레이션
    print(f"📦 재고 확인 완료! (주문: {order_id})")
    return f"재고 OK (주문: {order_id})"

if __name__ == '__main__':
    print("📦 재고 서비스 시작 (포트: 5001)")
    app.run(host='0.0.0.0', port=5001)
```

#### sync-version/shipping/app.py

```python
# 배송 서비스 (처리 시간: 3초 - 의도적으로 느림!)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/schedule')
def schedule_shipping():
    order_id = request.args.get('order_id', 'unknown')
    print(f"🚚 배송 예약 중... (주문: {order_id}) - 3초 소요")
    time.sleep(3)  # 외부 배송사 API 시뮬레이션 (느림!)
    print(f"🚚 배송 예약 완료! (주문: {order_id})")
    return f"배송 OK (주문: {order_id})"

if __name__ == '__main__':
    print("🚚 배송 서비스 시작 (포트: 5002)")
    app.run(host='0.0.0.0', port=5002)
```

#### sync-version/notification/app.py

```python
# 알림 서비스 (처리 시간: 0.3초)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/send')
def send_notification():
    order_id = request.args.get('order_id', 'unknown')
    print(f"📱 알림 발송 중... (주문: {order_id})")
    time.sleep(0.3)  # SMS/푸시 발송 시뮬레이션
    print(f"📱 알림 발송 완료! (주문: {order_id})")
    return f"알림 OK (주문: {order_id})"

if __name__ == '__main__':
    print("📱 알림 서비스 시작 (포트: 5003)")
    app.run(host='0.0.0.0', port=5003)
```

#### sync-version/Dockerfile (모든 서비스 공통)

```dockerfile
FROM python:3.9-slim
WORKDIR /app
RUN pip install flask requests
COPY app.py .
CMD ["python", "app.py"]
```

#### sync-version/compose.yml

```yaml
# ============================================================
# 동기식 주문 시스템
# 문제: 모든 서비스 순차 호출 → 총 처리 시간 ~4초
# ============================================================

services:
  order:
    build: ./order
    ports:
      - "5000:5000"
    depends_on:
      - inventory
      - shipping
      - notification
    environment:
      - PYTHONUNBUFFERED=1

  inventory:
    build: ./inventory
    environment:
      - PYTHONUNBUFFERED=1

  shipping:
    build: ./shipping
    environment:
      - PYTHONUNBUFFERED=1

  notification:
    build: ./notification
    environment:
      - PYTHONUNBUFFERED=1
```

> 💡 **PYTHONUNBUFFERED=1**: Python의 print 출력이 버퍼링 없이 즉시 로그에 표시됨

#### 동기식 버전 실행 및 테스트

```bash
cd sync-version

# 빌드 및 실행
docker compose up --build -d

# 주문 테스트 (jq로 결과 확인)
curl -s -X POST http://localhost:5000/order | jq .

# 로그 확인
docker compose logs -f order

# 정리
docker compose down
```

**예상 결과** (~4초 소요):
```json
{
  "status": "completed",
  "order_id": "a1b2c3d4",
  "elapsed_seconds": 3.89,
  "message": "주문 완료! (처리 시간: 3.9초)"
}
```

### 6.3 비동기식 버전 (Kafka 적용)

**비동기식의 핵심 변화**:
- 주문 서비스는 Kafka에 메시지 발행 후 **즉시 응답**
- 각 Consumer는 **독립적**으로 자기 속도대로 처리
- 하나가 느려도/죽어도 다른 서비스에 영향 없음

#### async-version/order/app.py (Producer)

```python
# ============================================================
# 비동기식 주문 서비스 (Kafka Producer)
# 핵심: HTTP 호출 대신 Kafka 메시지 발행 → 즉시 응답
# ============================================================
from flask import Flask, jsonify, request
from confluent_kafka import Producer
import json
import time
import uuid

app = Flask(__name__)

# Kafka Producer 설정
kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'client.id': 'order-service-producer',
    'acks': 'all',
    'retries': 3,
    'retry.backoff.ms': 100,
    'linger.ms': 0,  # 즉시 전송
}

producer = Producer(kafka_config)

def delivery_report(err, msg):
    if err is not None:
        print(f'❌ 메시지 전송 실패: {err}')
    else:
        print(f'✅ 메시지 전송 성공: partition={msg.partition()}, offset={msg.offset()}')

@app.route('/order', methods=['POST'])
def create_order():
    start = time.time()
    order_id = str(uuid.uuid4())[:8]

    print(f"\n{'='*50}")
    print(f"🛒 [비동기식] 주문 접수: {order_id}")
    print(f"{'='*50}")

    # 주문 데이터 준비
    order_data = {
        "order_id": order_id,
        "timestamp": time.time(),
        "status": "received",
        "product": request.json.get('product', '샘플 상품') if request.is_json else '샘플 상품',
        "quantity": request.json.get('quantity', 1) if request.is_json else 1,
    }

    # Kafka로 메시지 발행
    try:
        print(f"📤 Kafka로 메시지 발행 중...")
        producer.produce(
            topic='orders',
            key=order_id.encode('utf-8'),
            value=json.dumps(order_data).encode('utf-8'),
            callback=delivery_report
        )
        producer.poll(0)
        producer.flush(timeout=5)
        print(f"✅ 메시지 발행 완료!")
    except Exception as e:
        print(f"❌ Kafka 발행 실패: {e}")
        return jsonify({"status": "error", "message": "주문 접수 실패"}), 500

    elapsed = time.time() - start
    print(f"\n⏱️ API 응답 시간: {elapsed:.3f}초")

    # 핵심: "completed"가 아니라 "accepted"
    return jsonify({
        "status": "accepted",
        "order_id": order_id,
        "elapsed_seconds": round(elapsed, 3),
        "message": f"주문이 접수되었습니다. (처리 시간: {elapsed:.3f}초)",
        "note": "실제 처리는 백그라운드에서 진행됩니다."
    })

@app.route('/health')
def health_check():
    return jsonify({"status": "healthy", "service": "order"})

if __name__ == '__main__':
    print("🚀 [비동기식] 주문 서비스 시작 (포트: 5000)")
    app.run(host='0.0.0.0', port=5000)
```

#### async-version/inventory/app.py (Consumer)

```python
# ============================================================
# 재고 서비스 (Kafka Consumer)
# 독립적으로 동작, 자기 속도대로 처리
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'inventory-service-group',
    'client.id': 'inventory-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"📦 재고 확인 시작: 주문={order_id}")
    time.sleep(0.5)  # DB 조회 시뮬레이션
    print(f"📦 ✅ 재고 확인 완료: 주문={order_id}")

def main():
    print("="*60)
    print("📦 재고 서비스 (Kafka Consumer) 시작")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/shipping/app.py (Consumer - 느린 서비스)

```python
# ============================================================
# 배송 서비스 (Kafka Consumer) - 3초 걸리는 느린 서비스
# 핵심: 이 서비스가 느려도 주문 서비스와 다른 Consumer에 영향 없음!
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'shipping-service-group',
    'client.id': 'shipping-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"🚚 배송 예약 시작: 주문={order_id}")
    print(f"🚚 ⏳ 외부 배송사 API 호출 중... (3초 소요)")
    time.sleep(3)  # 느린 외부 API 시뮬레이션
    print(f"🚚 ✅ 배송 예약 완료: 주문={order_id}")

def main():
    print("="*60)
    print("🚚 배송 서비스 (Kafka Consumer) 시작")
    print("⚠️  주의: 메시지당 3초 소요")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/notification/app.py (Consumer)

```python
# ============================================================
# 알림 서비스 (Kafka Consumer)
# 빠른 처리 (0.3초), 실패해도 주문 자체는 성공
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'notification-service-group',
    'client.id': 'notification-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"📱 알림 발송 시작: 주문={order_id}")
    time.sleep(0.3)  # SMS/푸시 발송 시뮬레이션
    print(f"📱 ✅ 알림 발송 완료: 주문={order_id}")

def main():
    print("="*60)
    print("📱 알림 서비스 (Kafka Consumer) 시작")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/Dockerfile (모든 서비스 공통)

```dockerfile
FROM python:3.9-slim
WORKDIR /app

# confluent-kafka 설치를 위한 시스템 의존성
RUN apt-get update && apt-get install -y \
    gcc \
    librdkafka-dev \
    && rm -rf /var/lib/apt/lists/*

# Python 패키지 설치
RUN pip install flask confluent-kafka

COPY app.py .
CMD ["python", "-u", "app.py"]
```

#### async-version/compose.yml

```yaml
# ============================================================
# 비동기식 주문 시스템 (Kafka 기반)
#
# 핵심 변화:
# - order 서비스: Kafka에 발행 후 즉시 응답 (~0.05초)
# - 각 Consumer: 독립적으로 자기 속도대로 처리
# ============================================================

services:
  # Kafka 브로커 (KRaft 모드)
  kafka:
    image: apache/kafka:latest
    container_name: kafka-broker
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      KAFKA_NUM_PARTITIONS: 3
      KAFKA_DEFAULT_REPLICATION_FACTOR: 1
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
    healthcheck:
      test: ["CMD-SHELL", "/opt/kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 || exit 1"]
      interval: 10s
      timeout: 10s
      retries: 5
      start_period: 30s

  # 주문 서비스 (Producer)
  order:
    build: ./order
    container_name: order-service
    ports:
      - "5000:5000"
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 재고 서비스 (Consumer)
  inventory:
    build: ./inventory
    container_name: inventory-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 배송 서비스 (Consumer) - 느림!
  shipping:
    build: ./shipping
    container_name: shipping-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 알림 서비스 (Consumer)
  notification:
    build: ./notification
    container_name: notification-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # Kafka UI (모니터링용)
  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    ports:
      - "8080:8080"
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
```

---
## 7. jq 실전 활용

### 동기식 vs 비동기식 응답 비교

```bash
# 동기식 응답 (jq로 예쁘게 출력)
curl -s -X POST http://localhost:5000/order | jq .
# {
#   "status": "completed",
#   "elapsed_seconds": 3.89
# }

# 비동기식 응답
curl -s -X POST http://localhost:5000/order | jq .
# {
#   "status": "accepted",
#   "elapsed_seconds": 0.052
# }
```

### 특정 필드만 추출

```bash
# 응답 시간만 추출
curl -s -X POST http://localhost:5000/order | jq '.elapsed_seconds'

# 메시지만 추출 (따옴표 없이)
curl -s -X POST http://localhost:5000/order | jq -r '.message'

# 여러 필드 추출
curl -s -X POST http://localhost:5000/order | jq '{status, elapsed_seconds}'
```

### 시간 측정과 함께 사용

```bash
# time 명령어와 함께 사용
time curl -s -X POST http://localhost:5000/order | jq .
```

---
## 8. 비교 실습 및 정리

### 8.1 실습 A: 응답 시간 비교

```bash
# === 동기식 테스트 ===
cd sync-version
docker compose up -d
sleep 5

echo "=== 동기식 주문 테스트 ==="
time curl -s -X POST http://localhost:5000/order | jq .

docker compose down

# === 비동기식 테스트 ===
cd ../async-version
docker compose up -d
sleep 30  # Kafka 준비 대기

echo "=== 비동기식 주문 테스트 ==="
time curl -s -X POST http://localhost:5000/order | jq .

docker compose logs -f
```

### 8.2 결과 비교표

| 항목 | 동기식 | 비동기식 |
|------|--------|----------|
| **API 응답 시간** | ~4초 | ~0.05초 |
| **사용자 대기 시간** | 4초 | 즉시 |
| **응답 의미** | "모든 처리 완료" | "접수됨 (처리 중)" |
| **status 값** | `completed` | `accepted` |

### 8.3 실습 B: 장애 격리 비교

```bash
# === 동기식에서 배송 장애 ===
cd sync-version
docker compose up -d
docker compose stop shipping
curl -s -X POST http://localhost:5000/order | jq .
# 결과: {"error": "배송 서비스 연결 실패"} - 전체 실패!

# === 비동기식에서 배송 장애 ===
cd ../async-version
docker compose up -d
sleep 30
docker compose stop shipping
curl -s -X POST http://localhost:5000/order | jq .
# 결과: {"status": "accepted"} - 주문 접수 성공!

# 배송 서비스 복구
docker compose start shipping
docker compose logs -f shipping
# 결과: 밀린 메시지가 순차적으로 처리됨!
```

### 8.4 장애 격리 비교표

| 상황 | 동기식 | 비동기식 |
|------|--------|----------|
| **배송 장애 시 주문** | ❌ 전체 실패 | ✅ 접수 성공 |
| **장애 중 메시지** | 유실 | Kafka에 보관 |
| **복구 후** | 수동 재처리 필요 | 자동으로 처리 |

### 8.5 비동기식의 한계점

| 한계 | 설명 | 해결책 |
|------|------|--------|
| **"완료" 의미 변화** | API 응답 = "접수됨" (처리 완료 아님) | 별도 상태 조회 API, 웹소켓/SSE |
| **에러 핸들링 복잡** | Consumer 실패 시 클라이언트가 모름 | Dead Letter Queue, 보상 트랜잭션 |
| **순서 보장 어려움** | 재고/배송이 순서대로 처리 안될 수 있음 | 단일 Consumer, 이벤트 체이닝 |
| **트랜잭션 처리** | 부분 실패 가능 | Saga 패턴 |

### 8.6 언제 무엇을 사용할까?

**동기식이 적합한 경우**:
- 즉각적인 결과 확인 필요 (결제 승인, 로그인)
- 트랜잭션 무결성 중요
- 서비스 수가 적고 안정적

**비동기식(Kafka)이 적합한 경우**:
- 처리 결과를 나중에 확인해도 됨
- 높은 처리량 필요
- 서비스 간 느슨한 결합 필요
- 장애 격리 중요

> 💡 **실제 서비스에서는 혼합 사용!** 예: 결제(동기) + 배송/알림(비동기)

---
## 📚 FAQ

**Q1. confluent-kafka와 kafka-python 중 어떤 것을 사용해야 하나요?**

A1. 프로덕션 환경에서는 `confluent-kafka`를 권장합니다. librdkafka C 라이브러리 기반으로 10배 이상 빠르고, Confluent 공식 지원을 받습니다. `kafka-python`은 순수 Python으로 설치가 쉽지만 성능이 낮아 학습용으로만 적합합니다.

**Q2. `PYTHONUNBUFFERED=1`은 왜 필요한가요?**

A2. Python은 기본적으로 stdout을 버퍼링합니다. Docker 환경에서 `print()` 출력이 즉시 로그에 나타나지 않을 수 있습니다. `PYTHONUNBUFFERED=1`을 설정하면 버퍼링 없이 즉시 출력됩니다.

**Q3. Consumer의 `group.id`를 왜 서비스마다 다르게 설정하나요?**

A3. 같은 `group.id`를 가진 Consumer들은 메시지를 **분담**해서 처리합니다. 다른 `group.id`를 사용하면 **모든 Consumer가 모든 메시지**를 받습니다. 재고/배송/알림이 각각 모든 주문을 처리해야 하므로 서로 다른 그룹을 사용합니다.

**Q4. jq에서 `-r` 옵션은 언제 사용하나요?**

A4. 기본적으로 jq는 문자열을 따옴표로 감싸서 출력합니다. `-r` (raw output) 옵션을 사용하면 따옴표 없이 순수 문자열로 출력됩니다. 스크립트에서 값을 변수에 저장할 때 유용합니다.

**Q5. 비동기식에서 "주문 완료"를 어떻게 확인하나요?**

A5. 별도의 상태 조회 API (`GET /order/{id}/status`)를 만들거나, 웹소켓/SSE로 실시간 알림을 보내거나, 모든 처리 완료 후 별도 이벤트를 발행하는 방식을 사용합니다.

---
## 📝 퀴즈

### Q1. confluent-kafka의 특징으로 올바른 것은?

- A) 순수 Python으로 작성되어 설치가 간편하다
- B) asyncio를 기본 지원하여 비동기 처리에 최적화되어 있다
- C) librdkafka C 라이브러리 기반으로 고성능을 제공한다
- D) Apache 재단에서 공식 지원하는 라이브러리이다

<details>
<summary>정답 확인</summary>

**정답: C**

confluent-kafka는 librdkafka C 라이브러리를 기반으로 하여 kafka-python보다 10배 이상 빠른 성능을 제공합니다. Confluent(Kafka 창시자 회사)에서 공식 지원합니다.
</details>

---

### Q2. 다음 jq 명령어 중 한글이 정상 출력되는 것은?

- A) `curl http://api | cat`
- B) `curl http://api | jq`
- C) `curl http://api | python -c "import json; print(json.load(sys.stdin))"`
- D) `curl http://api | grep message`

<details>
<summary>정답 확인</summary>

**정답: B**

jq는 JSON을 파싱하면서 유니코드 이스케이프 시퀀스(`\uXXXX`)를 실제 문자로 변환합니다. 다른 방법들은 유니코드가 그대로 출력되거나 추가 처리가 필요합니다.
</details>

---

### Q3. Kafka Consumer에서 `group.id`의 역할은?

- A) 메시지의 키를 지정하여 파티션을 결정한다
- B) 같은 그룹의 Consumer들이 메시지를 분담 처리하도록 한다
- C) Producer와 Consumer를 연결하는 식별자이다
- D) 토픽의 파티션 수를 결정한다

<details>
<summary>정답 확인</summary>

**정답: B**

같은 `group.id`를 가진 Consumer들은 Consumer Group을 형성하여 파티션을 나눠서 처리합니다. 다른 그룹의 Consumer들은 같은 메시지를 각각 독립적으로 처리합니다.
</details>

---

### Q4. 비동기식 주문 시스템에서 `status: "accepted"` 응답의 의미는?

- A) 주문이 완료되어 배송이 시작되었다
- B) 모든 서비스(재고, 배송, 알림)가 처리를 완료했다
- C) 주문이 Kafka에 저장되었고, 백그라운드 처리가 진행 중이다
- D) 결제가 승인되어 재고가 차감되었다

<details>
<summary>정답 확인</summary>

**정답: C**

비동기식에서 `accepted`는 "주문 메시지가 Kafka에 저장됨"을 의미합니다. 실제 처리(재고 확인, 배송 예약, 알림)는 각 Consumer가 독립적으로 진행합니다.
</details>

---
## ✏️ 과제

### 과제 1: 동기식 vs 비동기식 응답 시간 비교 (난이도: ⭐)

1. 동기식 버전 실행 후 주문 API 호출, 응답 시간 기록
2. 비동기식 버전 실행 후 주문 API 호출, 응답 시간 기록
3. 두 결과를 비교하고 차이점 분석

<details>
<summary>💡 힌트</summary>

```bash
# 시간 측정과 jq 함께 사용
time curl -s -X POST http://localhost:5000/order | jq .
```
</details>

---

### 과제 2: 장애 격리 테스트 (난이도: ⭐⭐)

1. 비동기식 버전 실행
2. 배송 서비스 중지: `docker compose stop shipping`
3. 주문 3개 생성
4. 재고/알림 서비스 로그 확인 (처리 완료 여부)
5. 배송 서비스 재시작 후 밀린 메시지 처리 확인

<details>
<summary>💡 힌트</summary>

```bash
# 특정 서비스 로그만 확인
docker compose logs -f inventory
docker compose logs -f shipping
```
</details>

---

### 과제 3: jq 활용 스크립트 작성 (난이도: ⭐⭐)

연속 5개 주문을 생성하고, 각 응답의 `elapsed_seconds`만 추출하여 평균 계산

<details>
<summary>💡 힌트</summary>

```bash
# 반복문으로 주문 생성, 응답 시간만 추출
for i in {1..5}; do
  curl -s -X POST http://localhost:5000/order | jq '.elapsed_seconds'
done
```
</details>

---

### 보너스 과제: Kafka UI에서 메시지 확인 (난이도: ⭐⭐⭐)

1. 비동기식 버전 실행 후 `http://localhost:8080` 접속
2. `orders` 토픽 확인
3. 주문 생성 후 메시지 내용 확인
4. 각 Consumer Group의 오프셋 상태 확인
5. Consumer 하나를 중지했다 재시작하며 Lag 변화 관찰

---
## 🎯 핵심 요약

### 1. 포트 매핑
- Docker 컨테이너와 호스트는 별도 네트워크
- `-p 호스트:컨테이너`로 연결

### 2. jq 도구
- JSON 포맷팅 및 필드 추출
- 유니코드 깨짐 해결: `curl ... | jq .`

### 3. confluent-kafka
- librdkafka 기반 고성능 라이브러리
- Producer: `produce()` → `flush()`
- Consumer: `subscribe()` → `poll()` → `close()`

### 4. 동기식 vs 비동기식

| 항목 | 동기식 | 비동기식 |
|------|--------|----------|
| 응답 시간 | ~4초 | ~0.05초 |
| 응답 의미 | 완료 | 접수됨 |
| 장애 영향 | 전파 | 격리 |
| 메시지 보존 | 없음 | Kafka에 저장 |

### 5. 실제 적용
- 동기식: 결제, 로그인 등 즉각 확인 필요한 경우
- 비동기식: 배송, 알림 등 나중에 처리해도 되는 경우
- **실무에서는 혼합 사용!**